In [4]:
import pandas as pd
import numpy as np

# 1. Read file - using header=0 to avoid the 'Timestamp' string error
df = pd.read_excel('Book4.xlsx', usecols="B,C,H", header=0) 
df.columns = ['Timestamp', 'Ex1_Close', 'Ex2_Close']

# 2. Convert and Clean
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
# Ensure the CLOSE columns are numeric
df['Ex1_Close'] = pd.to_numeric(df['Ex1_Close'], errors='coerce')
df['Ex2_Close'] = pd.to_numeric(df['Ex2_Close'], errors='coerce')

# 3. Drop any rows with NaT or NaN
df = df.dropna()

# 4. Filter for 13:00 to 19:00
mask = (df['Timestamp'].dt.hour == 13) & (df['Timestamp'].dt.hour == 19)
filtered_data = df.loc[mask]

# 5. Check if we actually have data left
if len(filtered_data) > 1:
    slope, intercept = np.polyfit(filtered_data['Ex1_Close'], 
                                  filtered_data['Ex2_Close'], 1)
    
    print(f"--- Results for 13:00 - 19:00 ---")
    print(f"Total data points: {len(filtered_data)}")
    print(f"Slope (m):         {slope:.6f}")
    print(f"Intercept (c):     {intercept:.6f}")
else:
    print("Error: Not enough data points found in the 13:00-19:00 window.")

Error: Not enough data points found in the 13:00-19:00 window.


C:\Users\Axxela\AppData\Local\Temp\ipykernel_29972\1142780820.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')


In [5]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import date, timedelta

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'Book4.xlsx'
OUTPUT_SHEET   = 'Spread_Analysis'
EXCLUDED_DATES = {date(2026, 4, 3)}
TARGET_TIMES   = [(13, 0), (19, 0)]   # exact HH:MM only

SD_COLOURS = {
     4: ("1E3A10", True),
     3: ("375623", True),
     2: ("70AD47", False),
     1: ("C6EFCE", False),
    -1: ("FFEB9C", False),
    -2: ("FF9C00", False),
    -3: ("FF0000", True),
    -4: ("7B0000", True),
}

# ─────────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────────
df = pd.read_excel(INPUT_FILE, usecols="B,C,H", header=0)
df.columns = ['Timestamp', 'Ex1_Close', 'Ex2_Close']
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')
df['Ex1_Close'] = pd.to_numeric(df['Ex1_Close'], errors='coerce')
df['Ex2_Close'] = pd.to_numeric(df['Ex2_Close'], errors='coerce')
df = df.dropna()

# ─────────────────────────────────────────────
# 2. LAST 30 WORKDAYS — counted backwards from latest date in data
# ─────────────────────────────────────────────
latest_date = df['Timestamp'].dt.date.max()
print(f"Latest date in data: {latest_date}")

def last_n_workdays(n, from_date, excluded=set()):
    workdays = []
    d = from_date
    while len(workdays) < n:
        if d.weekday() < 5 and d not in excluded:
            workdays.append(d)
        d -= timedelta(days=1)
    return set(workdays)

valid_dates = last_n_workdays(30, latest_date, excluded=EXCLUDED_DATES)
print(f"Date window: {min(valid_dates)} → {max(valid_dates)}")
print(f"Valid trading days: {len(valid_dates)}")

# ─────────────────────────────────────────────
# 3. FILTER: exact 13:00 and 19:00 rows only
# ─────────────────────────────────────────────
mask = (
    df['Timestamp'].dt.date.isin(valid_dates) &
    df[['Timestamp']].apply(
        lambda row: (row['Timestamp'].hour, row['Timestamp'].minute) in TARGET_TIMES,
        axis=1
    )
)
filtered = df.loc[mask].copy().sort_values('Timestamp').reset_index(drop=True)
print(f"\nData points found at 13:00 & 19:00: {len(filtered)}")

if len(filtered) < 2:
    raise ValueError(
        "Not enough data points. Check TARGET_TIMES matches your data exactly.\n"
        f"Hours available: {sorted(df['Timestamp'].dt.hour.unique())}"
    )

# ─────────────────────────────────────────────
# 4. REGRESSION + RESIDUALS + SD BANDS
# ─────────────────────────────────────────────
x = filtered['Ex1_Close'].values
y = filtered['Ex2_Close'].values

slope, intercept = np.polyfit(x, y, 1)
y_hat      = slope * x + intercept
residuals  = y - y_hat
res_mean   = residuals.mean()
res_std    = residuals.std(ddof=1)

filtered['Predicted_Ex2'] = y_hat.round(4)
filtered['Residual']      = residuals.round(4)
filtered['Residual_Z']    = ((residuals - res_mean) / res_std).round(3)

def sd_bucket(z):
    for threshold in [4, 3, 2, 1]:
        if z >=  threshold: return  threshold
        if z <= -threshold: return -threshold
    return 0

filtered['SD_Band'] = filtered['Residual_Z'].apply(sd_bucket)

print(f"\nSlope     : {slope:.6f}")
print(f"Intercept : {intercept:.6f}")
print(f"Residual μ: {res_mean:.6f}")
print(f"Residual σ: {res_std:.6f}")
print(f"\nSD Band counts:\n{filtered['SD_Band'].value_counts().sort_index()}")

# ─────────────────────────────────────────────
# 5. WRITE TO EXCEL — delete old sheet first
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

output_cols = ['Timestamp', 'Ex1_Close', 'Ex2_Close', 'Predicted_Ex2', 'Residual', 'Residual_Z']
summary_rows = 9  # rows reserved at top for summary block

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    filtered[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 6. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

# ── Summary block ──
summary_data = [
    ("SPREAD ANALYSIS — Last 30 Working Days",),
    (f"Period: {min(valid_dates)}  →  {max(valid_dates)}",),
    (f"Snapshots: 13:00 & 19:00  |  Data points: {len(filtered)}  |  Excluded: Apr 3",),
    (),
    ("Metric",              "Value",            "Interpretation"),
    ("Slope  (m)",          f"{slope:.6f}",      "Units of Ex2 per unit of Ex1"),
    ("Intercept  (c)",      f"{intercept:.6f}",  "Ex2 value when Ex1 = 0"),
    ("Residual Mean  (μ)",  f"{res_mean:.6f}",   "Avg spread vs regression line"),
    ("Residual SD  (σ)",    f"{res_std:.6f}",    "1 SD move in the spread"),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 5:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx >= 6:
            cell.font = Font(bold=(c_idx == 1))

# ── SD legend (columns F-H, rows 5-9) ──
legend = [
    ("SD Band",  "Meaning",                    ""),
    ("+3/+4 SD", "Ex2 very expensive vs Ex1",  "375623"),
    ("+1/+2 SD", "Ex2 slightly expensive",     "70AD47"),
    ("-1/-2 SD", "Ex2 slightly cheap",         "FF9C00"),
    ("-3/-4 SD", "Ex2 very cheap vs Ex1",      "FF0000"),
]
for r_idx, (band, meaning, hex_c) in enumerate(legend, start=5):
    ce = ws.cell(row=r_idx, column=6, value=band)
    cf = ws.cell(row=r_idx, column=7, value=meaning)
    if r_idx == 5:
        ce.font = Font(bold=True)
        cf.font = Font(bold=True)
    elif hex_c:
        dark = hex_c in ("375623", "FF0000")
        pf   = PatternFill(fill_type="solid", fgColor=hex_c)
        for cell in (ce, cf):
            cell.fill = pf
            cell.font = Font(bold=True, color="FFFFFF" if dark else "000000")

# ── Header row ──
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# ── Row highlights ──
data_start = summary_rows + 2
for i, (sd_val, z_val) in enumerate(zip(filtered['SD_Band'], filtered['Residual_Z'])):
    if sd_val == 0:
        continue
    row_num     = data_start + i
    hex_c, dark = SD_COLOURS[sd_val]
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if dark else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# ── Column widths ──
for col, width in zip("ABCDEFG", [22, 13, 13, 15, 12, 12, 13, 28]):
    ws.column_dimensions[col].width = width

ws.freeze_panes = f"A{data_start + 1}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' sheet updated in {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_15580\2531058788.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')


Latest date in data: 2026-04-06
Date window: 2026-02-23 → 2026-04-06
Valid trading days: 30

Data points found at 13:00 & 19:00: 30

Slope     : 21.464473
Intercept : 2674.130648
Residual μ: 0.000000
Residual σ: 162.534209

SD Band counts:
SD_Band
-2     2
-1     4
 0    19
 1     5
Name: count, dtype: int64

✓ Done — 'Spread_Analysis' sheet updated in Book4.xlsx


In [4]:
import pandas as pd

df = pd.read_excel('Book4.xlsx', usecols="B,C,H", header=0)
df.columns = ['Timestamp', 'Ex1_Close', 'Ex2_Close']
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')
df = df.dropna()

# Show ALL unique hour:minute combos that exist
print("All unique times in data:")
times = df['Timestamp'].dt.strftime('%H:%M').value_counts().sort_index()
print(times.to_string())

All unique times in data:
Timestamp
00:00    24
00:01    24
00:02    24
00:03    24
00:04    24
00:05    24
00:06    24
00:07    24
00:08    24
00:09    24
00:10    24
00:11    24
00:12    24
00:13    24
00:14    24
00:15    24
00:16    24
00:17    24
00:18    24
00:19    24
00:20    24
00:21    24
00:22    24
00:23    24
00:24    24
00:25    24
00:26    24
00:27    24
00:28    24
00:29    24
00:30    24
00:31    24
00:32    24
00:33    24
00:34    24
00:35    24
00:36    24
00:37    24
00:38    24
00:39    24
00:40    24
00:41    24
00:42    24
00:43    24
00:44    24
00:45    24
00:46    24
00:47    24
00:48    24
00:49    24
00:50    24
00:51    24
00:52    24
00:53    24
00:54    24
00:55    24
00:56    24
00:57    24
00:58    24
00:59    24
01:00    24
01:01    24
01:02    24
01:03    24
01:04    24
01:05    24
01:06    24
01:07    24
01:08    24
01:09    24
01:10    24
01:11    24
01:12    24
01:13    24
01:14    24
01:15    24
01:16    24
01:17    24
01:18    24
01:19    24
01:2

C:\Users\Axxela\AppData\Local\Temp\ipykernel_15580\2061097251.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True, errors='coerce')


In [5]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'Book4.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v3'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120   # exactly 120 working days
DISPLAY_DAYS   = 60    # last 60 working days to show

BAND_COLOURS = {
    'Entry':  ('FF9C00', True),
    'In':     ('FFEB9C', False),
    'Exit':   ('FFFFCC', False),
    'Neutral':  (None,     False),
}

# ─────────────────────────────────────────────
# 1. LOAD RAW DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

# XAU: columns J (index 9) and K (index 10)
xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['XAU_Time', 'XAU_Close']
xau['XAU_Time']  = pd.to_datetime(xau['XAU_Time'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna(subset=['XAU_Time', 'XAU_Close'])
xau['Date'] = xau['XAU_Time'].dt.date
xau_daily = xau.groupby('Date').last().reset_index()[['Date', 'XAU_Close']]

# GDX: columns M (index 12) and N (index 13)
gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['GDX_Time', 'GDX_Close']
gdx['GDX_Time']  = pd.to_datetime(gdx['GDX_Time'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna(subset=['GDX_Time', 'GDX_Close'])
gdx['Date'] = gdx['GDX_Time'].dt.date
gdx_daily = gdx.groupby('Date').last().reset_index()[['Date', 'GDX_Close']]

# ─────────────────────────────────────────────
# 2. MERGE WITH FORWARD-FILL
# ─────────────────────────────────────────────
all_dates = pd.DataFrame({
    'Date': sorted(set(xau_daily['Date']) | set(gdx_daily['Date']))
})

merged = all_dates.merge(xau_daily, on='Date', how='left') \
                  .merge(gdx_daily, on='Date', how='left')

merged['XAU_Close'] = merged['XAU_Close'].ffill()
merged['GDX_Close'] = merged['GDX_Close'].ffill()
merged = merged.dropna().reset_index(drop=True)

# Keep only working days, exclude specified dates
merged = merged[
    merged['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

print(f"Total merged working-day rows: {len(merged)}")
print(f"Full date range: {merged['Date'].min()} → {merged['Date'].max()}")

# ─────────────────────────────────────────────
# 3. ROLLING 120 WORKING-DAY REGRESSION
#    merged is already weekday-only with excluded dates removed,
#    so each row IS a working day — rolling over 120 rows = 120 working days
# ─────────────────────────────────────────────
slopes     = []
intercepts = []
residuals  = []
z_scores   = []

x_all = merged['XAU_Close'].values
y_all = merged['GDX_Close'].values

for i in range(len(merged)):
    if i < ROLLING_WINDOW - 1:
        # Not enough working-day history yet
        slopes.append(np.nan)
        intercepts.append(np.nan)
        residuals.append(np.nan)
        z_scores.append(np.nan)
        continue

    x_win = x_all[i - ROLLING_WINDOW + 1 : i + 1]
    y_win = y_all[i - ROLLING_WINDOW + 1 : i + 1]

    m, c   = np.polyfit(x_win, y_win, 1)
    y_hat  = m * x_win + c
    res    = y_win - y_hat
    res_mu = res.mean()
    res_sd = res.std(ddof=1)

    current_res = y_all[i] - (m * x_all[i] + c)
    z = (current_res - res_mu) / res_sd if res_sd != 0 else 0.0

    slopes.append(round(m, 6))
    intercepts.append(round(c, 6))
    residuals.append(round(current_res, 4))
    z_scores.append(round(z, 4))

merged['Slope']     = slopes
merged['Intercept'] = intercepts
merged['Residual']  = residuals
merged['Z_Score']   = z_scores

# ─────────────────────────────────────────────
# 4. ENTRY / EXIT STATE MACHINE
# Band 1: Entry Z < -2,  Exit Z > -1
# Band 2: Entry Z < -3,  Exit Z > -2
# Band 3: Entry Z < -4,  Exit Z > -3
# ─────
def compute_signals(z_series):
    states = []

    in_trade   = False
    entry_z    = None
    exit_level = None
    band       = 0

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            continue

        # ─────────────────────────
        # ENTRY (only if flat)
        # ─────────────────────────
        if not in_trade:
            if z < -4:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -3:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -2:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -1:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            else:
                states.append('Neutral')

        # ─────────────────────────
        # IN TRADE
        # ─────────────────────────
        else:
            if z >= exit_level:
                states.append(f'Exit')
                in_trade   = False
                entry_z    = None
                exit_level = None
                band       = 0
            else:
                states.append(f'In')

    return states


merged['Signal'] = compute_signals(merged['Z_Score'].values)

# ─────────────────────────────────────────────
# 5. FILTER TO LAST 60 WORKING DAYS FOR DISPLAY
# ─────────────────────────────────────────────
latest = merged['Date'].max()

def last_n_workdays(n, from_date, excluded=set()):
    days = []
    d = from_date
    while len(days) < n:
        if d.weekday() < 5 and d not in excluded:
            days.append(d)
        d -= timedelta(days=1)
    return set(days)

display_dates = last_n_workdays(DISPLAY_DAYS, latest, EXCLUDED_DATES)
display = merged[merged['Date'].isin(display_dates)].copy() \
                .sort_values('Date').reset_index(drop=True)

# Verify no NaNs in display window
nan_count = display['Z_Score'].isna().sum()
print(f"\nDisplay rows (last {DISPLAY_DAYS} working days): {len(display)}")
print(f"Display range: {display['Date'].min()} → {display['Date'].max()}")
print(f"Rows with NaN Z_Score in display window: {nan_count}")
if nan_count > 0:
    print("WARNING: Some display rows have no regression — data doesn't go back far enough.")
    print(f"Need at least {ROLLING_WINDOW} working days before {display['Date'].min()}")
print(f"\nSignal counts:\n{display['Signal'].value_counts()}")

# ─────────────────────────────────────────────
# 6. WRITE TO EXCEL
# ─────────────────────────────────────────────
output_cols  = ['Date', 'XAU_Close', 'GDX_Close', 'Slope', 'Intercept', 'Residual', 'Z_Score', 'Signal']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 7. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

min_d = min(display_dates)
max_d = max(display_dates)

summary_data = [
    ("XAU / GDX SPREAD ANALYSIS — Rolling 120-Working-Day Regression",),
    (f"Display window : Last {DISPLAY_DAYS} working days  ({min_d} → {max_d})",),
    (f"Rolling window : {ROLLING_WINDOW} working days  |  Excluded: Apr 3 2026",),
    (),
    ("Band",            "Entry Trigger", "Exit Trigger", "Fill"),
    ("Band 1 ",   "Z < -1",        "Z > 0",       ""),
    ("Band 1 ",   "Z < -2",        "Z > -1",       ""),
    ("Band 2 ", "Z < -3",        "Z > -2",       ""),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 5:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx in (6, 7, 8):
            cell.font = Font(bold=(c_idx == 1))

# Band colour swatches
swatch_map = {6: 'FFEB9C', 7: 'FF9C00', 8: 'FF0000'}
for row_num, hex_c in swatch_map.items():
    cell = ws.cell(row=row_num, column=4, value="          ")
    cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)

# Header row
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# Row highlights
data_start = summary_rows + 2
for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c is None:
        continue
    row_num = data_start + i
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if white else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# Column widths
col_widths = [14, 13, 13, 13, 14, 12, 12, 14]
for i, width in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = width

ws.freeze_panes = f"A{data_start}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' written to {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\3795281638.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['XAU_Time']  = pd.to_datetime(xau['XAU_Time'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\3795281638.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['GDX_Time']  = pd.to_datetime(gdx['GDX_Time'], dayfirst=True, errors='coerce')


Total merged working-day rows: 206
Full date range: 2025-06-30 → 2026-04-14

Display rows (last 60 working days): 60
Display range: 2026-01-20 → 2026-04-14
Rows with NaN Z_Score in display window: 0

Signal counts:
Signal
Neutral    37
In         17
Entry       3
Exit        3
Name: count, dtype: int64

✓ Done — 'Spread_Analysis_v3' written to Book4.xlsx


In [3]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'Book4.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
EXCLUDED_DATES = {date(2026, 4, 3)}

TIME_WINDOW = pd.Timedelta('120D')   # TRUE rolling window (quant correct)
DISPLAY_ROWS = 60 * 24

BAND_COLOURS = {
    'Entry':  ('FF9C00', True),
    'In':     ('FFEB9C', False),
    'Exit':   ('FFFFCC', False),
    'Neutral':  (None, False),
}

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime'] = pd.to_datetime(xau['DateTime'], errors='coerce', dayfirst=True)
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime')

gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime'] = pd.to_datetime(gdx['DateTime'], errors='coerce', dayfirst=True)
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna().sort_values('DateTime')

# ─────────────────────────────────────────────
# 2. LOG TRANSFORM (CRITICAL FOR STATIONARITY)
# ─────────────────────────────────────────────
xau['XAU_Log'] = np.log(xau['XAU_Close'])
gdx['GDX_Log'] = np.log(gdx['GDX_Close'])

# ─────────────────────────────────────────────
# 3. TIME ALIGNMENT (NO FAKE GRID)
# ─────────────────────────────────────────────
merged = pd.merge_asof(
    xau.sort_values('DateTime'),
    gdx.sort_values('DateTime'),
    on='DateTime',
    direction='backward',
    tolerance=pd.Timedelta('2h')
)

merged = merged.dropna().reset_index(drop=True)
merged = merged[merged['DateTime'].dt.weekday < 5].reset_index(drop=True)

# replace with log series
merged['XAU'] = np.log(merged['XAU_Close'])
merged['GDX'] = np.log(merged['GDX_Close'])

print(f"Total rows: {len(merged)}")
print(f"Range: {merged['DateTime'].min()} → {merged['DateTime'].max()}")

# ─────────────────────────────────────────────
# 4. TRUE TIME-BASED ROLLING REGRESSION
# ─────────────────────────────────────────────
slopes, intercepts, residuals, z_scores = [], [], [], []

for i in range(len(merged)):
    current_time = merged['DateTime'].iloc[i]

    window_mask = merged['DateTime'] >= (current_time - TIME_WINDOW)

    x_win = merged.loc[window_mask, 'XAU'].values
    y_win = merged.loc[window_mask, 'GDX'].values

    if len(x_win) < 50:
        slopes.append(np.nan)
        intercepts.append(np.nan)
        residuals.append(np.nan)
        z_scores.append(np.nan)
        continue

    # regression
    m, c = np.polyfit(x_win, y_win, 1)

    y_hat = m * x_win + c
    res = y_win - y_hat

    current_res = merged.loc[i, 'GDX'] - (m * merged.loc[i, 'XAU'] + c)

    res_mu = res.mean()

    # robust volatility estimator (quant-grade)
    res_sd = np.percentile(np.abs(res - res_mu), 68)

    z = (current_res - res_mu) / res_sd if res_sd != 0 else 0.0

    slopes.append(round(m, 6))
    intercepts.append(round(c, 6))
    residuals.append(round(current_res, 6))
    z_scores.append(round(z, 4))

merged['Slope'] = slopes
merged['Intercept'] = intercepts
merged['Residual'] = residuals
merged['Z_Score'] = z_scores

# ─────────────────────────────────────────────
# 5. CLEAN SIGNAL ENGINE (SYMMETRIC MEAN REVERSION)
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states = []

    in_trade   = False
    entry_z    = None
    exit_level = None
    band       = 0

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            continue

        # ─────────────────────────
        # ENTRY (only if flat)
        # ─────────────────────────
        if not in_trade:
            if z < -4:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -3:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -2:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            elif z < -1:
                in_trade   = True
                entry_z    = z
                exit_level = z + 1
                band       = 1
                states.append('Entry')

            else:
                states.append('Neutral')

        # ─────────────────────────
        # IN TRADE
        # ─────────────────────────
        else:
            if z >= exit_level:
                states.append(f'Exit')
                in_trade   = False
                entry_z    = None
                exit_level = None
                band       = 0
            else:
                states.append(f'In')

    return states

merged['Signal'] = compute_signals(merged['Z_Score'].values)

# ─────────────────────────────────────────────
# 6. DISPLAY WINDOW
# ─────────────────────────────────────────────
display = merged.tail(DISPLAY_ROWS).reset_index(drop=True)

print("\nDisplay rows:", len(display))
print("Range:", display['DateTime'].min(), "→", display['DateTime'].max())
print("\nZ-score stats:")
print(display['Z_Score'].describe())

# ─────────────────────────────────────────────
# 7. EXPORT TO EXCEL
# ─────────────────────────────────────────────
output_cols = ['DateTime','XAU_Close','GDX_Close','Slope','Intercept','Residual','Z_Score','Signal']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 8. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

header_row = summary_rows + 1

for col in range(1, len(output_cols)+1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill("solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

data_start = summary_rows + 2

for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if not hex_c:
        continue

    row = data_start + i
    fill = PatternFill("solid", fgColor=hex_c)
    font = Font(color="FFFFFF" if white else "000000")

    for col in range(1, len(output_cols)+1):
        cell = ws.cell(row=row, column=col)
        cell.fill = fill
        cell.font = font

ws.column_dimensions['A'].width = 20
ws.freeze_panes = f"A{data_start}"

wb.save(INPUT_FILE)

print("\n✓ TRUE quant stat-arb model complete")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\2871788495.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime'] = pd.to_datetime(xau['DateTime'], errors='coerce', dayfirst=True)
C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\2871788495.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime'] = pd.to_datetime(gdx['DateTime'], errors='coerce', dayfirst=True)


Total rows: 3198
Range: 2025-06-30 23:30:00 → 2026-04-13 23:30:00

Display rows: 1440
Range: 2025-12-02 06:30:00 → 2026-04-13 23:30:00

Z-score stats:
count    1440.000000
mean       -0.010746
std         1.001431
min        -3.267600
25%        -0.656900
50%         0.242600
75%         0.680700
max         1.718400
Name: Z_Score, dtype: float64

✓ TRUE quant stat-arb model complete


In [14]:
print(merged[['XAU_Close','GDX_Close']].describe())
print("GDX unique values:", merged['GDX_Close'].nunique())

         XAU_Close    GDX_Close
count  3198.000000  3198.000000
mean   4166.091477    79.351402
std     615.509671    17.457494
min    3269.100000    50.490000
25%    3631.452500    67.216250
50%    4142.450000    78.790000
75%    4647.053225    93.955000
max    5546.320000   119.200000
GDX unique values: 2390


In [7]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'Book4.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120   # working days of daily closes
DISPLAY_DAYS   = 60    # last N working days of hourly bars to show

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),
    'In_3':     ('FF0000', True),
    'Exit_3':   ('FF6600', True),
    'Entry_2':  ('FF0000', True),
    'In_2':     ('FF9C00', True),
    'Exit_2':   ('FFCC00', False),
    'Entry_1':  ('FF9C00', True),
    'In_1':     ('FFEB9C', False),
    'Exit_1':   ('FFFFCC', False),
    'Neutral':  (None,     False),
}

# ─────────────────────────────────────────────
# 1. LOAD RAW HOURLY DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

# XAU: columns J (index 9) and K (index 10)
xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)
xau['Date'] = xau['DateTime'].dt.date

# GDX: columns M (index 12) and N (index 13)
gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna().sort_values('DateTime').reset_index(drop=True)
gdx['Date'] = gdx['DateTime'].dt.date

print(f"XAU hourly rows: {len(xau)}  |  range: {xau['DateTime'].min()} → {xau['DateTime'].max()}")
print(f"GDX hourly rows: {len(gdx)}  |  range: {gdx['DateTime'].min()} → {gdx['DateTime'].max()}")

# ─────────────────────────────────────────────
# 2. DAILY CLOSES (for regression)
# ─────────────────────────────────────────────
xau_daily = xau.groupby('Date')['XAU_Close'].last().reset_index()
gdx_daily = gdx.groupby('Date')['GDX_Close'].last().reset_index()

# Merge daily closes — forward fill if one side missing on a day
all_dates = pd.DataFrame({
    'Date': sorted(set(xau_daily['Date']) | set(gdx_daily['Date']))
})
daily = all_dates.merge(xau_daily, on='Date', how='left') \
                 .merge(gdx_daily, on='Date', how='left')
daily['XAU_Close'] = daily['XAU_Close'].ffill()
daily['GDX_Close'] = daily['GDX_Close'].ffill()
daily = daily.dropna().reset_index(drop=True)

# Keep only working days, remove excluded dates
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

print(f"\nDaily working-day rows: {len(daily)}")
print(f"Daily range: {daily['Date'].min()} → {daily['Date'].max()}")

# ─────────────────────────────────────────────
# 3. ROLLING 120 WORKING-DAY REGRESSION ON DAILY CLOSES
# ─────────────────────────────────────────────
x_all = daily['XAU_Close'].values
y_all = daily['GDX_Close'].values

d_slopes     = []
d_intercepts = []
d_residuals  = []
d_zscores    = []

for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        d_slopes.append(np.nan)
        d_intercepts.append(np.nan)
        d_residuals.append(np.nan)
        d_zscores.append(np.nan)
        continue

    x_win = x_all[i - ROLLING_WINDOW + 1 : i + 1]
    y_win = y_all[i - ROLLING_WINDOW + 1 : i + 1]

    m, c    = np.polyfit(x_win, y_win, 1)
    y_hat   = m * x_win + c
    res     = y_win - y_hat
    res_mu  = res.mean()
    res_sd  = res.std(ddof=1)

    current_res = y_all[i] - (m * x_all[i] + c)
    z = (current_res - res_mu) / res_sd if res_sd != 0 else 0.0

    d_slopes.append(round(m, 6))
    d_intercepts.append(round(c, 6))
    d_residuals.append(round(current_res, 4))
    d_zscores.append(round(z, 4))

daily['Slope']     = d_slopes
daily['Intercept'] = d_intercepts
daily['Residual']  = d_residuals
daily['Z_Score']   = d_zscores

print(f"\nDaily rows with valid Z_Score: {daily['Z_Score'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. ENTRY / EXIT STATE MACHINE ON DAILY Z-SCORES
# Band 1: Entry Z < -2,  Exit Z > -1
# Band 2: Entry Z < -3,  Exit Z > -2
# Band 3: Entry Z < -4,  Exit Z > -3
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states  = []
    in_band = 0

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            continue

        if in_band == 0:
            if z < -4:
                in_band = 3
                states.append('Entry_3')
            elif z < -3:
                in_band = 2
                states.append('Entry_2')
            elif z < -2:
                in_band = 1
                states.append('Entry_1')
            else:
                states.append('Neutral')

        elif in_band == 1:
            if z < -3:
                in_band = 2
                states.append('Entry_2')
            elif z > -1:
                in_band = 0
                states.append('Exit_1')
            else:
                states.append('In_1')

        elif in_band == 2:
            if z < -4:
                in_band = 3
                states.append('Entry_3')
            elif z > -2:
                in_band = 1 if z < -1 else 0
                states.append('Exit_2')
            else:
                states.append('In_2')

        elif in_band == 3:
            if z > -3:
                in_band = 2 if z < -2 else (1 if z < -1 else 0)
                states.append('Exit_3')
            else:
                states.append('In_3')

    return states

daily['Signal'] = compute_signals(daily['Z_Score'].values)

# ─────────────────────────────────────────────
# 5. STAMP DAILY STATS ONTO HOURLY ROWS
#    Every hourly bar in a given date gets that
#    date's slope, intercept, residual, z, signal
# ─────────────────────────────────────────────
daily_lookup = daily.set_index('Date')[['Slope','Intercept','Residual','Z_Score','Signal']]

# Build combined hourly frame from both instruments
# Use XAU timestamps as the hourly spine, merge GDX by nearest timestamp
xau_h = xau.copy()
gdx_h = gdx[['DateTime','GDX_Close']].copy()

hourly = pd.merge_asof(
    xau_h.sort_values('DateTime'),
    gdx_h.sort_values('DateTime'),
    on='DateTime',
    direction='nearest',
    tolerance=pd.Timedelta('2h')
)

hourly = hourly.dropna(subset=['GDX_Close']).reset_index(drop=True)
hourly['Date'] = hourly['DateTime'].dt.date

# Keep only working days
hourly = hourly[
    hourly['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

# Stamp daily regression values onto each hourly row
hourly = hourly.join(daily_lookup, on='Date')

print(f"\nHourly rows after merge: {len(hourly)}")

# ─────────────────────────────────────────────
# 6. FILTER TO LAST 60 WORKING DAYS OF HOURLY BARS
# ─────────────────────────────────────────────
latest = hourly['Date'].max()

def last_n_workdays(n, from_date, excluded=set()):
    days = []
    d = from_date
    while len(days) < n:
        if d.weekday() < 5 and d not in excluded:
            days.append(d)
        d -= timedelta(days=1)
    return set(days)

display_dates = last_n_workdays(DISPLAY_DAYS, latest, EXCLUDED_DATES)
display = hourly[hourly['Date'].isin(display_dates)].copy() \
                .sort_values('DateTime').reset_index(drop=True)

nan_count = display['Z_Score'].isna().sum()
print(f"Display hourly rows (last {DISPLAY_DAYS} working days): {len(display)}")
print(f"Display range: {display['Date'].min()} → {display['Date'].max()}")
print(f"Rows with NaN Z_Score: {nan_count}")
if nan_count > 0:
    print("WARNING: Data doesn't go back far enough — add history from ~Jul 2025")
print(f"\nSignal counts:\n{display['Signal'].value_counts()}")

# ─────────────────────────────────────────────
# 7. WRITE TO EXCEL
# ─────────────────────────────────────────────
output_cols  = ['DateTime', 'XAU_Close', 'GDX_Close', 'Slope', 'Intercept', 'Residual', 'Z_Score', 'Signal']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 8. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

min_d = min(display_dates)
max_d = max(display_dates)

summary_data = [
    ("XAU / GDX SPREAD ANALYSIS — Rolling 120-Working-Day Regression",),
    (f"Display window : Last {DISPLAY_DAYS} working days of hourly bars  ({min_d} → {max_d})",),
    (f"Regression     : Daily closes, rolling {ROLLING_WINDOW} working days  |  Excluded: Apr 3 2026",),
    (f"Signal stamped : Same day's regression values applied to all intraday bars",),
    (),
    ("Band",            "Entry Trigger", "Exit Trigger", "Fill"),
    ("Band 1 — Mild",   "Z < -2",        "Z > -1",       ""),
    ("Band 2 — Medium", "Z < -3",        "Z > -2",       ""),
    ("Band 3 — Strong", "Z < -4",        "Z > -3",       ""),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 6:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx in (7, 8, 9):
            cell.font = Font(bold=(c_idx == 1))

# Band colour swatches
swatch_map = {7: 'FFEB9C', 8: 'FF9C00', 9: 'FF0000'}
for row_num, hex_c in swatch_map.items():
    cell = ws.cell(row=row_num, column=4, value="          ")
    cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)

# Header row
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# Row highlights
data_start = summary_rows + 2
for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c is None:
        continue
    row_num = data_start + i
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if white else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# Column widths
col_widths = [20, 13, 13, 13, 14, 12, 12, 14]
for i, width in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = width

ws.freeze_panes = f"A{data_start}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' written to {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\979973371.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_22396\979973371.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')


XAU hourly rows: 4827  |  range: 2025-06-30 23:30:00 → 2026-04-13 23:30:00
GDX hourly rows: 3146  |  range: 2025-06-30 23:30:00 → 2026-04-14 00:30:00

Daily working-day rows: 206
Daily range: 2025-06-30 → 2026-04-14

Daily rows with valid Z_Score: 87

Hourly rows after merge: 3588
Display hourly rows (last 60 working days): 1049
Display range: 2026-01-20 → 2026-04-13
Rows with NaN Z_Score: 0

Signal counts:
Signal
Neutral    802
In_1       192
Entry_1     32
Exit_1      23
Name: count, dtype: int64

✓ Done — 'Spread_Analysis_v2' written to Book4.xlsx


In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'GDX_XAU_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120   # working days of daily closes
DISPLAY_DAYS   = 60    # last N working days of hourly bars to show

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),
    'In_3':     ('FF0000', True),
    'Exit_3':   ('FF6600', True),
    'Entry_2':  ('FF0000', True),
    'In_2':     ('FF9C00', True),
    'Exit_2':   ('FFCC00', False),
    'Entry_1':  ('FF9C00', True),
    'In_1':     ('FFEB9C', False),
    'Exit_1':   ('FFFFCC', False),
    'Neutral':  (None,     False),
}

# ─────────────────────────────────────────────
# 1. LOAD RAW HOURLY DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

# XAU: columns J (index 9) and K (index 10)
xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)
xau['Date'] = xau['DateTime'].dt.date

# GDX: columns M (index 12) and N (index 13)
gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna().sort_values('DateTime').reset_index(drop=True)
gdx['Date'] = gdx['DateTime'].dt.date

print(f"XAU hourly rows: {len(xau)}  |  range: {xau['DateTime'].min()} → {xau['DateTime'].max()}")
print(f"GDX hourly rows: {len(gdx)}  |  range: {gdx['DateTime'].min()} → {gdx['DateTime'].max()}")

# ─────────────────────────────────────────────
# 2. DAILY CLOSES (for regression)
# ─────────────────────────────────────────────
xau_daily = xau.groupby('Date')['XAU_Close'].last().reset_index()
gdx_daily = gdx.groupby('Date')['GDX_Close'].last().reset_index()

# Merge daily closes — forward fill if one side missing on a day
all_dates = pd.DataFrame({
    'Date': sorted(set(xau_daily['Date']) | set(gdx_daily['Date']))
})
daily = all_dates.merge(xau_daily, on='Date', how='left') \
                 .merge(gdx_daily, on='Date', how='left')
daily['XAU_Close'] = daily['XAU_Close'].ffill()
daily['GDX_Close'] = daily['GDX_Close'].ffill()
daily = daily.dropna().reset_index(drop=True)

# Keep only working days, remove excluded dates
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

print(f"\nDaily working-day rows: {len(daily)}")
print(f"Daily range: {daily['Date'].min()} → {daily['Date'].max()}")

# ─────────────────────────────────────────────
# 3. ROLLING 120 WORKING-DAY REGRESSION ON DAILY CLOSES
# ─────────────────────────────────────────────
x_all = daily['XAU_Close'].values
y_all = daily['GDX_Close'].values

d_slopes     = []
d_intercepts = []
d_res_means  = []
d_res_stds   = []

for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        d_slopes.append(np.nan)
        d_intercepts.append(np.nan)
        d_res_means.append(np.nan)
        d_res_stds.append(np.nan)
        continue

    x_win = x_all[i - ROLLING_WINDOW + 1 : i + 1]
    y_win = y_all[i - ROLLING_WINDOW + 1 : i + 1]

    m, c   = np.polyfit(x_win, y_win, 1)
    y_hat  = m * x_win + c
    res    = y_win - y_hat

    d_slopes.append(round(m, 6))
    d_intercepts.append(round(c, 6))
    d_res_means.append(res.mean())
    d_res_stds.append(res.std(ddof=1))

daily['Slope']     = d_slopes
daily['Intercept'] = d_intercepts
daily['Res_Mean']  = d_res_means
daily['Res_Std']   = d_res_stds

print(f"\nDaily rows with valid Slope: {daily['Slope'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. ENTRY / EXIT STATE MACHINE (runs on hourly Z-scores in step 5)
# Band 1: Entry Z < -2,  Exit Z > -1
# Band 2: Entry Z < -3,  Exit Z > -2
# Band 3: Entry Z < -4,  Exit Z > -3
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states  = []
    in_band = 0

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            continue

        if in_band == 0:
            if z < -4:
                in_band = 3
                states.append('Entry_3')
            elif z < -3:
                in_band = 2
                states.append('Entry_2')
            elif z < -2:
                in_band = 1
                states.append('Entry_1')
            else:
                states.append('Neutral')

        elif in_band == 1:
            if z < -3:
                in_band = 2
                states.append('Entry_2')
            elif z > -1:
                in_band = 0
                states.append('Exit_1')
            else:
                states.append('In_1')

        elif in_band == 2:
            if z < -4:
                in_band = 3
                states.append('Entry_3')
            elif z > -2:
                in_band = 1 if z < -1 else 0
                states.append('Exit_2')
            else:
                states.append('In_2')

        elif in_band == 3:
            if z > -3:
                in_band = 2 if z < -2 else (1 if z < -1 else 0)
                states.append('Exit_3')
            else:
                states.append('In_3')

    return states

# ─────────────────────────────────────────────
# 5. STAMP SLOPE/INTERCEPT ONTO HOURLY ROWS
#    Then compute residual and Z-score per hourly bar
#    using that hour's actual XAU/GDX prices
# ─────────────────────────────────────────────

# Only stamp slope & intercept from daily — NOT residual/Z/signal
daily_lookup = daily.set_index('Date')[['Slope', 'Intercept']]

# Also keep rolling window residual stats (mean & std) for Z normalisation
# These are computed from the 120-day window at each daily point
res_stats = daily.set_index('Date')[['Res_Mean', 'Res_Std']]

# Build combined hourly frame from both instruments
# Use XAU timestamps as the hourly spine, merge GDX by nearest timestamp
xau_h = xau.copy()
gdx_h = gdx[['DateTime','GDX_Close']].copy()

hourly = pd.merge_asof(
    xau_h.sort_values('DateTime'),
    gdx_h.sort_values('DateTime'),
    on='DateTime',
    direction='nearest',
    tolerance=pd.Timedelta('2h')
)

hourly = hourly.dropna(subset=['GDX_Close']).reset_index(drop=True)
hourly['Date'] = hourly['DateTime'].dt.date

# Keep only working days
hourly = hourly[
    hourly['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

# Stamp slope, intercept, res_mean, res_std onto each hourly row
hourly = hourly.join(daily_lookup, on='Date')
hourly = hourly.join(res_stats,    on='Date')

# Compute residual and Z-score per hourly bar using that hour's prices
hourly['Residual'] = hourly['GDX_Close'] - (hourly['Slope'] * hourly['XAU_Close'] + hourly['Intercept'])
hourly['Z_Score']  = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']
hourly['Z_Score']  = hourly['Z_Score'].round(4)
hourly['Residual'] = hourly['Residual'].round(4)

# Compute signal per hourly bar based on hourly Z-score
hourly['Signal'] = compute_signals(hourly['Z_Score'].values)

print(f"\nHourly rows after merge: {len(hourly)}")

# ─────────────────────────────────────────────
# 6. FILTER TO LAST 60 WORKING DAYS OF HOURLY BARS
# ─────────────────────────────────────────────
latest = hourly['Date'].max()

def last_n_workdays(n, from_date, excluded=set()):
    days = []
    d = from_date
    while len(days) < n:
        if d.weekday() < 5 and d not in excluded:
            days.append(d)
        d -= timedelta(days=1)
    return set(days)

display_dates = last_n_workdays(DISPLAY_DAYS, latest, EXCLUDED_DATES)
display = hourly[hourly['Date'].isin(display_dates)].copy() \
                .sort_values('DateTime').reset_index(drop=True)

nan_count = display['Z_Score'].isna().sum()
print(f"Display hourly rows (last {DISPLAY_DAYS} working days): {len(display)}")
print(f"Display range: {display['Date'].min()} → {display['Date'].max()}")
print(f"Rows with NaN Z_Score: {nan_count}")
if nan_count > 0:
    print("WARNING: Data doesn't go back far enough — add history from ~Jul 2025")
print(f"\nSignal counts:\n{display['Signal'].value_counts()}")

# ─────────────────────────────────────────────
# 7. WRITE TO EXCEL
# ─────────────────────────────────────────────
output_cols  = ['DateTime', 'XAU_Close', 'GDX_Close', 'Slope', 'Intercept', 'Residual', 'Z_Score', 'Signal']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 8. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

min_d = min(display_dates)
max_d = max(display_dates)

summary_data = [
    ("XAU / GDX SPREAD ANALYSIS — Rolling 120-Working-Day Regression",),
    (f"Display window : Last {DISPLAY_DAYS} working days of hourly bars  ({min_d} → {max_d})",),
    (f"Regression     : Daily closes, rolling {ROLLING_WINDOW} working days  |  Excluded: Apr 3 2026",),
    (f"Signal stamped : Same day's regression values applied to all intraday bars",),
    (),
    ("Band",            "Entry Trigger", "Exit Trigger", "Fill"),
    ("Band 1 — Mild",   "Z < -2",        "Z > -1",       ""),
    ("Band 2 — Medium", "Z < -3",        "Z > -2",       ""),
    ("Band 3 — Strong", "Z < -4",        "Z > -3",       ""),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 6:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx in (7, 8, 9):
            cell.font = Font(bold=(c_idx == 1))

# Band colour swatches
swatch_map = {7: 'FFEB9C', 8: 'FF9C00', 9: 'FF0000'}
for row_num, hex_c in swatch_map.items():
    cell = ws.cell(row=row_num, column=4, value="          ")
    cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)

# Header row
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# Row highlights
data_start = summary_rows + 2
for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c is None:
        continue
    row_num = data_start + i
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if white else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# Column widths
col_widths = [20, 13, 13, 13, 14, 12, 12, 14]
for i, width in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = width

ws.freeze_panes = f"A{data_start}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' written to {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_25764\3517779600.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_25764\3517779600.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')


XAU hourly rows: 4827  |  range: 2025-06-30 23:30:00 → 2026-04-13 23:30:00
GDX hourly rows: 3146  |  range: 2025-06-30 23:30:00 → 2026-04-14 00:30:00

Daily working-day rows: 206
Daily range: 2025-06-30 → 2026-04-14

Daily rows with valid Slope: 87

Hourly rows after merge: 3588
Display hourly rows (last 60 working days): 1049
Display range: 2026-01-20 → 2026-04-13
Rows with NaN Z_Score: 0

Signal counts:
Signal
Neutral    805
In_1       200
In_2        33
Exit_1       4
Entry_1      3
Entry_2      2
Exit_2       2
Name: count, dtype: int64

✓ Done — 'Spread_Analysis_v2' written to GDX_XAU_spreadanalysis.xlsx


In [3]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'GDX_XAU_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120   # working days of daily closes
DISPLAY_DAYS   = 60    # last N working days of hourly bars to show

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),
    'In_3':     ('FF0000', True),
    'Exit_3':   ('FF6600', True),
    'Entry_2':  ('FF0000', True),
    'In_2':     ('FF9C00', True),
    'Exit_2':   ('FFCC00', False),
    'Entry_1':  ('FF9C00', True),
    'In_1':     ('FFEB9C', False),
    'Exit_1':   ('FFFFCC', False),
    'Neutral':  (None,     False),
}

# ─────────────────────────────────────────────
# 1. LOAD RAW HOURLY DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

# XAU: columns J (index 9) and K (index 10)
xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)
xau['Date'] = xau['DateTime'].dt.date

# GDX: columns M (index 12) and N (index 13)
gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna().sort_values('DateTime').reset_index(drop=True)
gdx['Date'] = gdx['DateTime'].dt.date

print(f"XAU hourly rows: {len(xau)}  |  range: {xau['DateTime'].min()} → {xau['DateTime'].max()}")
print(f"GDX hourly rows: {len(gdx)}  |  range: {gdx['DateTime'].min()} → {gdx['DateTime'].max()}")

# ─────────────────────────────────────────────
# 2. DAILY CLOSES (for regression)
# ─────────────────────────────────────────────
xau_daily = xau.groupby('Date')['XAU_Close'].last().reset_index()
gdx_daily = gdx.groupby('Date')['GDX_Close'].last().reset_index()

# Merge daily closes — forward fill if one side missing on a day
all_dates = pd.DataFrame({
    'Date': sorted(set(xau_daily['Date']) | set(gdx_daily['Date']))
})
daily = all_dates.merge(xau_daily, on='Date', how='left') \
                 .merge(gdx_daily, on='Date', how='left')
daily['XAU_Close'] = daily['XAU_Close'].ffill()
daily['GDX_Close'] = daily['GDX_Close'].ffill()
daily = daily.dropna().reset_index(drop=True)

# Keep only working days, remove excluded dates
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

print(f"\nDaily working-day rows: {len(daily)}")
print(f"Daily range: {daily['Date'].min()} → {daily['Date'].max()}")

# ─────────────────────────────────────────────
# 3. ROLLING 120 WORKING-DAY REGRESSION ON DAILY CLOSES
# ─────────────────────────────────────────────
x_all = daily['XAU_Close'].values
y_all = daily['GDX_Close'].values

d_slopes     = []
d_intercepts = []
d_res_means  = []
d_res_stds   = []

for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        d_slopes.append(np.nan)
        d_intercepts.append(np.nan)
        d_res_means.append(np.nan)
        d_res_stds.append(np.nan)
        continue

    x_win = x_all[i - ROLLING_WINDOW + 1 : i + 1]
    y_win = y_all[i - ROLLING_WINDOW + 1 : i + 1]

    m, c   = np.polyfit(x_win, y_win, 1)
    y_hat  = m * x_win + c
    res    = y_win - y_hat

    d_slopes.append(round(m, 6))
    d_intercepts.append(round(c, 6))
    d_res_means.append(res.mean())
    d_res_stds.append(res.std(ddof=1))

daily['Slope']     = d_slopes
daily['Intercept'] = d_intercepts
daily['Res_Mean']  = d_res_means
daily['Res_Std']   = d_res_stds

print(f"\nDaily rows with valid Slope: {daily['Slope'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. ENTRY / EXIT STATE MACHINE (runs on hourly Z-scores in step 5)
# Band 1: Entry Z < -2,  Exit Z > -1
# Band 2: Entry Z < -3,  Exit Z > -2
# Band 3: Entry Z < -4,  Exit Z > -3
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states   = []
    in_band  = 0      # 0 = flat, 1/2/3 = active band
    exit_lvl = None   # Z level that triggers exit for active band

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            continue

        if in_band == 0:
            # Look for first trigger — whichever is crossed first
            if z < -4:
                in_band  = 3
                exit_lvl = z+1
                states.append('Entry_3')
            elif z < -3:
                in_band  = 2
                exit_lvl = z+1
                states.append('Entry_2')
            elif z < -2:
                in_band  = 1
                exit_lvl = z+1
                states.append('Entry_1')
            elif z < -1:
                in_band  = 1
                exit_lvl = z+1
                states.append('Entry_1')
            else:
                states.append('Neutral')

        else:
            # In a trade — ignore all new entry signals
            # Only check exit for THIS band
            if z > exit_lvl:
                states.append(f'Exit_{in_band}')
                in_band  = 0
                exit_lvl = None
            else:
                states.append(f'In_{in_band}')

    return states

# ─────────────────────────────────────────────
# 5. STAMP SLOPE/INTERCEPT ONTO HOURLY ROWS
#    Then compute residual and Z-score per hourly bar
#    using that hour's actual XAU/GDX prices
# ─────────────────────────────────────────────

# Only stamp slope & intercept from daily — NOT residual/Z/signal
daily_lookup = daily.set_index('Date')[['Slope', 'Intercept']]

# Also keep rolling window residual stats (mean & std) for Z normalisation
# These are computed from the 120-day window at each daily point
res_stats = daily.set_index('Date')[['Res_Mean', 'Res_Std']]

# Build combined hourly frame from both instruments
# Use XAU timestamps as the hourly spine, merge GDX by nearest timestamp
xau_h = xau.copy()
gdx_h = gdx[['DateTime','GDX_Close']].copy()

hourly = pd.merge_asof(
    xau_h.sort_values('DateTime'),
    gdx_h.sort_values('DateTime'),
    on='DateTime',
    direction='nearest',
    tolerance=pd.Timedelta('2h')
)

hourly = hourly.dropna(subset=['GDX_Close']).reset_index(drop=True)
hourly['Date'] = hourly['DateTime'].dt.date

# Keep only working days
hourly = hourly[
    hourly['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

# Stamp slope, intercept, res_mean, res_std onto each hourly row
hourly = hourly.join(daily_lookup, on='Date')
hourly = hourly.join(res_stats,    on='Date')

# Compute residual and Z-score per hourly bar using that hour's prices
hourly['Residual'] = hourly['GDX_Close'] - (hourly['Slope'] * hourly['XAU_Close'] + hourly['Intercept'])
hourly['Z_Score']  = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']
hourly['Z_Score']  = hourly['Z_Score'].round(4)
hourly['Residual'] = hourly['Residual'].round(4)

# Compute signal per hourly bar based on hourly Z-score
hourly['Signal'] = compute_signals(hourly['Z_Score'].values)

print(f"\nHourly rows after merge: {len(hourly)}")

# ─────────────────────────────────────────────
# 6. FILTER TO LAST 60 WORKING DAYS OF HOURLY BARS
# ─────────────────────────────────────────────
latest = hourly['Date'].max()

def last_n_workdays(n, from_date, excluded=set()):
    days = []
    d = from_date
    while len(days) < n:
        if d.weekday() < 5 and d not in excluded:
            days.append(d)
        d -= timedelta(days=1)
    return set(days)

display_dates = last_n_workdays(DISPLAY_DAYS, latest, EXCLUDED_DATES)
display = hourly[hourly['Date'].isin(display_dates)].copy() \
                .sort_values('DateTime').reset_index(drop=True)

nan_count = display['Z_Score'].isna().sum()
print(f"Display hourly rows (last {DISPLAY_DAYS} working days): {len(display)}")
print(f"Display range: {display['Date'].min()} → {display['Date'].max()}")
print(f"Rows with NaN Z_Score: {nan_count}")
if nan_count > 0:
    print("WARNING: Data doesn't go back far enough — add history from ~Jul 2025")
print(f"\nSignal counts:\n{display['Signal'].value_counts()}")

# ─────────────────────────────────────────────
# 7. WRITE TO EXCEL
# ─────────────────────────────────────────────
output_cols  = ['DateTime', 'XAU_Close', 'GDX_Close', 'Slope', 'Intercept', 'Residual', 'Z_Score', 'Signal']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 8. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

min_d = min(display_dates)
max_d = max(display_dates)

summary_data = [
    ("XAU / GDX SPREAD ANALYSIS — Rolling 120-Working-Day Regression",),
    (f"Display window : Last {DISPLAY_DAYS} working days of hourly bars  ({min_d} → {max_d})",),
    (f"Regression     : Daily closes, rolling {ROLLING_WINDOW} working days  |  Excluded: Apr 3 2026",),
    (f"Signal stamped : Same day's regression values applied to all intraday bars",),
    (),
    ("Band",            "Entry Trigger", "Exit Trigger", "Fill"),
    ("Band 1 — Mild",   "Z < -2",        "Z > -1",       ""),
    ("Band 2 — Medium", "Z < -3",        "Z > -2",       ""),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 6:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx in (7, 8, 9):
            cell.font = Font(bold=(c_idx == 1))

# Band colour swatches
swatch_map = {7: 'FFEB9C', 8: 'FF9C00', 9: 'FF0000'}
for row_num, hex_c in swatch_map.items():
    cell = ws.cell(row=row_num, column=4, value="          ")
    cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)

# Header row
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# Row highlights
data_start = summary_rows + 2
for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c is None:
        continue
    row_num = data_start + i
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if white else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# Column widths
col_widths = [20, 13, 13, 13, 14, 12, 12, 14]
for i, width in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = width

ws.freeze_panes = f"A{data_start}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' written to {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_25764\4259558391.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_25764\4259558391.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')


XAU hourly rows: 4827  |  range: 2025-06-30 23:30:00 → 2026-04-13 23:30:00
GDX hourly rows: 3146  |  range: 2025-06-30 23:30:00 → 2026-04-14 00:30:00

Daily working-day rows: 206
Daily range: 2025-06-30 → 2026-04-14

Daily rows with valid Slope: 87

Hourly rows after merge: 3588
Display hourly rows (last 60 working days): 1049
Display range: 2026-01-20 → 2026-04-13
Rows with NaN Z_Score: 0

Signal counts:
Signal
Neutral    588
In_1       445
Entry_1      8
Exit_1       8
Name: count, dtype: int64

✓ Done — 'Spread_Analysis_v2' written to GDX_XAU_spreadanalysis.xlsx


In [2]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'GDX_XAU_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120   # working days of daily closes
DISPLAY_DAYS   = 60    # last N working days of hourly bars to show

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),
    'In_3':     ('FF0000', True),
    'Exit_3':   ('FF6600', True),
    'Entry_2':  ('FF0000', True),
    'In_2':     ('FF9C00', True),
    'Exit_2':   ('FFCC00', False),
    'Entry_1':  ('FF9C00', True),
    'In_1':     ('FFEB9C', False),
    'Exit_1':   ('FFFFCC', False),
    'Neutral':  (None,     False),
}

# ─────────────────────────────────────────────
# 1. LOAD RAW HOURLY DATA
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)

# XAU: columns J (index 9) and K (index 10)
xau = raw.iloc[:, [9, 10]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)
xau['Date'] = xau['DateTime'].dt.date

# GDX: columns M (index 12) and N (index 13)
gdx = raw.iloc[:, [12, 13]].copy()
gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')
gdx = gdx.dropna().sort_values('DateTime').reset_index(drop=True)
gdx['Date'] = gdx['DateTime'].dt.date

print(f"XAU hourly rows: {len(xau)}  |  range: {xau['DateTime'].min()} → {xau['DateTime'].max()}")
print(f"GDX hourly rows: {len(gdx)}  |  range: {gdx['DateTime'].min()} → {gdx['DateTime'].max()}")

# ─────────────────────────────────────────────
# 2. DAILY CLOSES (for regression)
# ─────────────────────────────────────────────
xau_daily = xau.groupby('Date')['XAU_Close'].last().reset_index()
gdx_daily = gdx.groupby('Date')['GDX_Close'].last().reset_index()

# Merge daily closes — forward fill if one side missing on a day
all_dates = pd.DataFrame({
    'Date': sorted(set(xau_daily['Date']) | set(gdx_daily['Date']))
})
daily = all_dates.merge(xau_daily, on='Date', how='left') \
                 .merge(gdx_daily, on='Date', how='left')
daily['XAU_Close'] = daily['XAU_Close'].ffill()
daily['GDX_Close'] = daily['GDX_Close'].ffill()
daily = daily.dropna().reset_index(drop=True)

# Keep only working days, remove excluded dates
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

print(f"\nDaily working-day rows: {len(daily)}")
print(f"Daily range: {daily['Date'].min()} → {daily['Date'].max()}")

# ─────────────────────────────────────────────
# 3. ROLLING 120 WORKING-DAY REGRESSION ON DAILY CLOSES
# ─────────────────────────────────────────────
x_all = daily['XAU_Close'].values
y_all = daily['GDX_Close'].values

d_slopes     = []
d_intercepts = []
d_res_means  = []
d_res_stds   = []

for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        d_slopes.append(np.nan)
        d_intercepts.append(np.nan)
        d_res_means.append(np.nan)
        d_res_stds.append(np.nan)
        continue

    x_win = x_all[i - ROLLING_WINDOW + 1 : i + 1]
    y_win = y_all[i - ROLLING_WINDOW + 1 : i + 1]

    m, c   = np.polyfit(x_win, y_win, 1)
    y_hat  = m * x_win + c
    res    = y_win - y_hat

    d_slopes.append(round(m, 6))
    d_intercepts.append(round(c, 6))
    d_res_means.append(res.mean())
    d_res_stds.append(res.std(ddof=1))

daily['Slope']     = d_slopes
daily['Intercept'] = d_intercepts
daily['Res_Mean']  = d_res_means
daily['Res_Std']   = d_res_stds

print(f"\nDaily rows with valid Slope: {daily['Slope'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. ENTRY / EXIT STATE MACHINE (Updated Logic)
# Band 1: Entry Z > -2,  Exit Z < -1
# Band 2: Entry Z > -3,  Exit Z < -2
# Band 3: Entry Z > -1,  Exit Z <  0
# ─────────────────────────────────────────────
def compute_signals(z_series, slope_series):
    states, gdx_lots, xau_lots = [], [], []
    
    # Track if band is active
    active = {1: False, 2: False, 3: False}
    
    # Threshold Definitions (Entry, Exit)
    # Note: Using your specific requests for Band 3
    logic = {
        1: {'entry': -2, 'exit': -1},
        2: {'entry': -3, 'exit': -2},
        3: {'entry': -1, 'exit':  0}
    }

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral'); gdx_lots.append(0); xau_lots.append(0)
            continue

        executed_exit = False
        
        # 1. CHECK FOR EXITS (Z < Exit Threshold)
        for b in [3, 2, 1]:
            if active[b]:
                if z < logic[b]['exit']:
                    active[b] = False
                    executed_exit = True
                    current_row_signal = f'Exit_{b}'
                    break # Exit one band at a time per row logic

        # 2. CHECK FOR ENTRIES (Z > Entry Threshold)
        if not executed_exit:
            # We check in order of priority/strength
            if z > logic[3]['entry'] and not active[3]:
                active[3] = True
                current_row_signal = 'Entry_3'
            elif z > logic[2]['entry'] and not active[2]:
                active[2] = True
                current_row_signal = 'Entry_2'
            elif z > logic[1]['entry'] and not active[1]:
                active[1] = True
                current_row_signal = 'Entry_1'
            else:
                # 3. LABEL "IN" OR "NEUTRAL"
                live_bands = [b for b in [3, 2, 1] if active[b]]
                if live_bands:
                    current_signal_val = max(live_bands)
                    current_row_signal = f'In_{current_signal_val}'
                else:
                    current_row_signal = 'Neutral'

        states.append(current_row_signal)
        
        # 4. SUM TOTAL LOTS
        num_active = sum(active.values())
        gdx_lots.append(360 * num_active)
        xau_lots.append(1 * num_active)

    return states, gdx_lots, xau_lots
# ─────────────────────────────────────────────
# 5. STAMP SLOPE/INTERCEPT ONTO HOURLY ROWS
#    Then compute residual and Z-score per hourly bar
#    using that hour's actual XAU/GDX prices
# ─────────────────────────────────────────────

# Only stamp slope & intercept from daily — NOT residual/Z/signal
daily_lookup = daily.set_index('Date')[['Slope', 'Intercept']]

# Also keep rolling window residual stats (mean & std) for Z normalisation
# These are computed from the 120-day window at each daily point
res_stats = daily.set_index('Date')[['Res_Mean', 'Res_Std']]

# Build combined hourly frame from both instruments
# Use XAU timestamps as the hourly spine, merge GDX by nearest timestamp
xau_h = xau.copy()
gdx_h = gdx[['DateTime','GDX_Close']].copy()

hourly = pd.merge_asof(
    xau_h.sort_values('DateTime'),
    gdx_h.sort_values('DateTime'),
    on='DateTime',
    direction='nearest',
    tolerance=pd.Timedelta('2h')
)

hourly = hourly.dropna(subset=['GDX_Close']).reset_index(drop=True)
hourly['Date'] = hourly['DateTime'].dt.date

# Keep only working days
hourly = hourly[
    hourly['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)
].reset_index(drop=True)

# Stamp slope, intercept, res_mean, res_std onto each hourly row
hourly = hourly.join(daily_lookup, on='Date')
hourly = hourly.join(res_stats,    on='Date')

# Compute residual and Z-score per hourly bar using that hour's prices
hourly['Residual'] = hourly['GDX_Close'] - (hourly['Slope'] * hourly['XAU_Close'] + hourly['Intercept'])
hourly['Z_Score']  = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']
hourly['Z_Score']  = hourly['Z_Score'].round(4)
hourly['Residual'] = hourly['Residual'].round(4)

signals, gdx_lots, xau_lots = compute_signals(hourly['Z_Score'].values, hourly['Slope'].values)
hourly['Signal']   = signals
hourly['GDX_Lots'] = gdx_lots
hourly['XAU_Lots'] = xau_lots

print(f"\nHourly rows after merge: {len(hourly)}")

# ─────────────────────────────────────────────
# 6. FILTER TO LAST 60 WORKING DAYS OF HOURLY BARS
# ─────────────────────────────────────────────
latest = hourly['Date'].max()

def last_n_workdays(n, from_date, excluded=set()):
    days = []
    d = from_date
    while len(days) < n:
        if d.weekday() < 5 and d not in excluded:
            days.append(d)
        d -= timedelta(days=1)
    return set(days)

display_dates = last_n_workdays(DISPLAY_DAYS, latest, EXCLUDED_DATES)
display = hourly[hourly['Date'].isin(display_dates)].copy() \
                .sort_values('DateTime').reset_index(drop=True)

nan_count = display['Z_Score'].isna().sum()
print(f"Display hourly rows (last {DISPLAY_DAYS} working days): {len(display)}")
print(f"Display range: {display['Date'].min()} → {display['Date'].max()}")
print(f"Rows with NaN Z_Score: {nan_count}")
if nan_count > 0:
    print("WARNING: Data doesn't go back far enough — add history from ~Jul 2025")
print(f"\nSignal counts:\n{display['Signal'].value_counts()}")

# ─────────────────────────────────────────────
# 7. WRITE TO EXCEL
# ─────────────────────────────────────────────
output_cols  = ['DateTime', 'XAU_Close', 'GDX_Close', 'Slope', 'Intercept', 'Residual', 'Z_Score', 'Signal', 'GDX_Lots', 'XAU_Lots']
summary_rows = 8

wb = load_workbook(INPUT_FILE)
if OUTPUT_SHEET in wb.sheetnames:
    del wb[OUTPUT_SHEET]
    wb.save(INPUT_FILE)
wb.close()

with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a') as writer:
    display[output_cols].to_excel(
        writer,
        sheet_name=OUTPUT_SHEET,
        index=False,
        startrow=summary_rows
    )

# ─────────────────────────────────────────────
# 8. FORMATTING
# ─────────────────────────────────────────────
wb = load_workbook(INPUT_FILE)
ws = wb[OUTPUT_SHEET]

min_d = min(display_dates)
max_d = max(display_dates)

summary_data = [
    ("XAU / GDX SPREAD ANALYSIS — Rolling 120-Working-Day Regression",),
    (f"Display window : Last {DISPLAY_DAYS} working days of hourly bars",),
    (f"Regression     : Daily closes, rolling {ROLLING_WINDOW} days",),
    (f"Signal stamped : Same day's regression values applied to intraday bars",),
    (),
    ("Band",            "Entry Trigger", "Exit Trigger", "Fill"),
    ("Band 1",          "Z > -2",        "Z < -1",       ""),
    ("Band 2",          "Z > -3",        "Z < -2",       ""),
    ("Band 3",          "Z > -1",        "Z < 0",        ""),
]

for r_idx, row_vals in enumerate(summary_data, start=1):
    for c_idx, val in enumerate(row_vals, start=1):
        cell = ws.cell(row=r_idx, column=c_idx, value=val)
        if r_idx == 1:
            cell.font = Font(bold=True, size=13, color="1F3864")
        elif r_idx == 6:
            cell.font = Font(bold=True, color="FFFFFF")
            cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
        elif r_idx in (7, 8, 9):
            cell.font = Font(bold=(c_idx == 1))

# Band colour swatches
swatch_map = {7: 'FFEB9C', 8: 'FF9C00', 9: 'FF0000'}
for row_num, hex_c in swatch_map.items():
    cell = ws.cell(row=row_num, column=4, value="          ")
    cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)

# Header row
header_row = summary_rows + 1
for col in range(1, len(output_cols) + 1):
    cell = ws.cell(row=header_row, column=col)
    cell.fill = PatternFill(fill_type="solid", fgColor="2F4F8F")
    cell.font = Font(bold=True, color="FFFFFF")
    cell.alignment = Alignment(horizontal='center')

# Row highlights
data_start = summary_rows + 2
for i, sig in enumerate(display['Signal']):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c is None:
        continue
    row_num = data_start + i
    pf  = PatternFill(fill_type="solid", fgColor=hex_c)
    fnt = Font(color="FFFFFF" if white else "000000")
    for col in range(1, len(output_cols) + 1):
        cell      = ws.cell(row=row_num, column=col)
        cell.fill = pf
        cell.font = fnt

# Column widths
col_widths = [20, 13, 13, 13, 14, 12, 12, 14, 11, 11]
for i, width in enumerate(col_widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = width

ws.freeze_panes = f"A{data_start}"
wb.save(INPUT_FILE)
print(f"\n✓ Done — '{OUTPUT_SHEET}' written to {INPUT_FILE}")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_26532\114975660.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_26532\114975660.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime']  = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')


XAU hourly rows: 4827  |  range: 2025-06-30 23:30:00 → 2026-04-13 23:30:00
GDX hourly rows: 3146  |  range: 2025-06-30 23:30:00 → 2026-04-14 00:30:00

Daily working-day rows: 206
Daily range: 2025-06-30 → 2026-04-14

Daily rows with valid Slope: 87

Hourly rows after merge: 3588
Display hourly rows (last 60 working days): 1049
Display range: 2026-01-20 → 2026-04-13
Rows with NaN Z_Score: 0

Signal counts:
Signal
In_3       423
Exit_3     156
Entry_3    156
Exit_1      86
Entry_1     86
Exit_2      70
Entry_2     70
Neutral      2
Name: count, dtype: int64

✓ Done — 'Spread_Analysis_v2' written to GDX_XAU_spreadanalysis.xlsx


--------------------------------

here

---------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'GDX_XAU_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
SUMMARY_SHEET  = 'Trade_Summary'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 120
DISPLAY_DAYS   = 60
PNL_MULTIPLIER = 10

BAND_COLOURS = {
    'Entry_4':  ('7B0000', True),  'Exit_4':   ('C6E0B4', False),
    'Entry_3':  ('C00000', True),  'Exit_3':   ('C6E0B4', False),
    'Entry_2':  ('FF0000', True),  'Exit_2':   ('C6E0B4', False),
    'Entry_1':  ('FF9C00', True),  'Exit_1':   ('C6E0B4', False),
}

LOTS_GDX = 360
LOTS_XAU = 1
BAND_THRESHOLDS = {1: -1, 2: -2, 3: -3, 4: -4}

# ─────────────────────────────────────────────
# 1. FIXED STATE MACHINE — Returns 3 values
# ─────────────────────────────────────────────
def compute_signals(z_series):
    signals = []
    gdx_lots = []
    xau_lots = []
    
    active = {1: False, 2: False, 3: False, 4: False}
    entry_z = {1: None, 2: None, 3: None, 4: None}

    for z in z_series:
        if np.isnan(z):
            signals.append('Neutral')
            gdx_lots.append(0)
            xau_lots.append(0)
            continue

        bar_events = []

        # 1. Check Exits (Z moves 1.0 point back)
        for b in [4, 3, 2, 1]:
            if active[b] and z >= (entry_z[b] + 1.0):
                active[b] = False
                entry_z[b] = None
                bar_events.append(f'Exit_{b}')

        # 2. Check Entries
        for b in [4, 3, 2, 1]:
            if not active[b] and z <= BAND_THRESHOLDS[b]:
                active[b] = True
                entry_z[b] = z
                bar_events.append(f'Entry_{b}')

        # 3. Determine Signal String
        if bar_events:
            sig_str = " | ".join(bar_events)
        else:
            live = [b for b in [4, 3, 2, 1] if active[b]]
            sig_str = f'In_{max(live)}' if live else 'Neutral'

        signals.append(sig_str)
        # Always append lots (or logic can be added here to scale lots)
        gdx_lots.append(LOTS_GDX)
        xau_lots.append(LOTS_XAU)

    return signals, gdx_lots, xau_lots

# ─────────────────────────────────────────────
# CALL SITE (Line 117)
# ─────────────────────────────────────────────
# This will now work without the Unpack error

# 2. DATA LOAD & PREP
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
xau = raw.iloc[:, [9, 10]].copy(); xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime'] = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')

gdx = raw.iloc[:, [12, 13]].copy(); gdx.columns = ['DateTime', 'GDX_Close']
gdx['DateTime'] = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')
gdx['GDX_Close'] = pd.to_numeric(gdx['GDX_Close'], errors='coerce')

xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)
gdx = gdx.dropna().sort_values('DateTime').reset_index(drop=True)

# Daily Regression
xau_d = xau.groupby(xau['DateTime'].dt.date)['XAU_Close'].last().reset_index()
gdx_d = gdx.groupby(gdx['DateTime'].dt.date)['GDX_Close'].last().reset_index()
daily = xau_d.merge(gdx_d, on='DateTime', how='inner')
daily.columns = ['Date', 'XAU_Close', 'GDX_Close']
daily = daily[daily['Date'].apply(lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES)].reset_index(drop=True)

slopes, ints, ms, ss = [], [], [], []
for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        for l in [slopes, ints, ms, ss]: l.append(np.nan)
        continue
    x = daily['XAU_Close'].values[i-ROLLING_WINDOW+1:i+1].astype(float)
    y = daily['GDX_Close'].values[i-ROLLING_WINDOW+1:i+1].astype(float)
    m, c = np.polyfit(x, y, 1)
    res = y - (m * x + c)
    slopes.append(m); ints.append(c); ms.append(res.mean()); ss.append(res.std(ddof=1))
daily['Slope'], daily['Intercept'], daily['Res_Mean'], daily['Res_Std'] = slopes, ints, ms, ss

# Hourly Merge
hourly = pd.merge_asof(xau, gdx, on='DateTime', direction='nearest')
hourly['Date'] = hourly['DateTime'].dt.date
hourly = hourly.join(daily.set_index('Date')[['Slope', 'Intercept', 'Res_Mean', 'Res_Std']], on='Date').dropna()
hourly['Residual'] = hourly['GDX_Close'] - (hourly['Slope'] * hourly['XAU_Close'] + hourly['Intercept'])
hourly['Z_Score']  = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']

sigs, gl, xl = compute_signals(hourly['Z_Score'].values)
hourly['Signal'], hourly['GDX_Lots'], hourly['XAU_Lots'] = sigs, gl, xl

# Filter to Display Window
start_dt = hourly['Date'].max() - timedelta(days=DISPLAY_DAYS)
display  = hourly[hourly['Date'] >= start_dt].copy().reset_index(drop=True)


# ─────────────────────────────────────────────
# 3. REVISED SUMMARY — Robust PnL & Multi-Event Logging
# ─────────────────────────────────────────────
summary_rows = []
vault = {1: None, 2: None, 3: None, 4: None}
active_sum = {1: False, 2: False, 3: False, 4: False}
entry_z_sum = {1: None, 2: None, 3: None, 4: None}

for _, row in display.iterrows():
    z = row['Z_Score']
    
    # Check Exits first so we can free up the band for a re-entry on same bar
    for b in [4, 3, 2, 1]:
        if active_sum[b] and z >= (entry_z_sum[b] + 1.0):
            en_g, en_x = vault[b]
            
            # PnL Calculation: Long GDX, Short XAU
            pnl_gdx = (row['GDX_Close'] - en_g) * LOTS_GDX
            pnl_xau = (en_x - row['XAU_Close']) * LOTS_XAU * PNL_MULTIPLIER
            
            r_exit = row.to_dict()
            r_exit['Band'] = b
            r_exit['Event'] = f'Exit_{b}'
            r_exit['Realized_PnL'] = round(pnl_gdx + pnl_xau, 2)
            summary_rows.append(r_exit)
            
            active_sum[b] = False
            vault[b] = None

    # Check Entries
    for b in [4, 3, 2, 1]:
        if not active_sum[b] and z <= BAND_THRESHOLDS[b]:
            active_sum[b] = True
            entry_z_sum[b] = z
            vault[b] = (row['GDX_Close'], row['XAU_Close'])
            
            r_entry = row.to_dict()
            r_entry['Band'] = b
            r_entry['Event'] = f'Entry_{b}'
            r_entry['Realized_PnL'] = 0.0
            summary_rows.append(r_entry)

summary_df = pd.DataFrame(summary_rows)
# Capture all 3 lists returned by the function
sigs, gl, xl = compute_signals(display['Z_Score'].values)

# Assign them to the correct columns
display['Signal'] = sigs
display['GDX_Lots'] = gl
display['XAU_Lots'] = xl
# ─────────────────────────────────────────────

# ─────────────────────────────────────────────
# 4. EXCEL OUTPUT
# ─────────────────────────────────────────────
with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    display.to_excel(writer, sheet_name=OUTPUT_SHEET, index=False, startrow=8)
    if not summary_df.empty:
        summary_df.to_excel(writer, sheet_name=SUMMARY_SHEET, index=False)

wb = load_workbook(INPUT_FILE)
ws_sum = wb[SUMMARY_SHEET]

# Apply Summary Styling
for i, event in enumerate(summary_df['Event'], start=2):
    hex_c, white = BAND_COLOURS.get(event, (None, False))
    if hex_c:
        for col in range(1, ws_sum.max_column + 1):
            cell = ws_sum.cell(row=i, column=col)
            cell.fill = PatternFill(fill_type="solid", fgColor=hex_c)
            if white:
                cell.font = Font(color="FFFFFF")

wb.save(INPUT_FILE)
print(f"✓ Process Complete. {len(summary_df)} trade events logged in {SUMMARY_SHEET}.")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_29396\1979727013.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime'] = pd.to_datetime(xau['DateTime'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_29396\1979727013.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdx['DateTime'] = pd.to_datetime(gdx['DateTime'], dayfirst=True, errors='coerce')


✓ Process Complete. 20 trade events logged in Trade_Summary.


26/26-27

In [8]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE      = 'WTI_spreadanalysis.xlsx'
SHEET_NAME      = 'Sheet1'
OUTPUT_SHEET    = 'Spread_Analysis_v2'
SUMMARY_SHEET   = 'Trade_Summary'
EXCLUDED_DATES  = set()          # add date(YYYY,M,D) entries if needed
ROLLING_WINDOW  = 30             # 30 calendar/business days
DISPLAY_DAYS    = 60             # last 60 working days shown
PNL_MULTIPLIER  = 1              # adjust as needed

# Lot sizes
LOTS_D26        = 2              # CLZ26 lots
LOTS_SPREAD     = 5              # CLZ26-Z27 (D26-D27) lots

# Price multipliers
MULT_D26        = 1000 * 2       # 2000 * Z26
MULT_SPREAD     = 1000 * 5       # 5000 * Z26-Z27

# Spread = 2000*Z26  -  5000*(Z26-Z27)
# i.e. long D26 side, short the calendar spread side

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),  'Exit_3':  ('C6E0B4', False),
    'Entry_2':  ('FF0000', True),  'Exit_2':  ('C6E0B4', False),
    'Entry_1':  ('FF9C00', True),  'Exit_1':  ('C6E0B4', False),
}

# ─────────────────────────────────────────────
# 1. STATE MACHINE
#    Entry:  z < -1 (Band 1), z < -2 (Band 2), z < -3 (Band 3)
#    Exit:   z > entry_z + 1  (i.e. +1 SD in favour of entry)
#    (entry_z is negative, so exit when z has moved +1 SD back toward mean from entry)
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states, d26_lots, spread_lots = [], [], []
    active   = {1: False, 2: False, 3: False}
    entry_z  = {1: None,  2: None,  3: None}

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral')
            d26_lots.append(0); spread_lots.append(0)
            continue

        sig = None

        # ── EXITS (deepest first) ──────────────────────────────
        for b in [3, 2, 1]:
            if active[b] and z > (entry_z[b] + 1):   # +1 SD in favour
                active[b]  = False
                entry_z[b] = None
                sig = f'Exit_{b}'
                break

        # ── ENTRIES (waterfall) ───────────────────────────────
        if sig is None:
            if z < -3 and not active[3]:
                active[3] = True; entry_z[3] = z
                if not active[2]: active[2] = True; entry_z[2] = z
                if not active[1]: active[1] = True; entry_z[1] = z
                sig = 'Entry_3'

            elif z < -2 and not active[2]:
                active[2] = True; entry_z[2] = z
                if not active[1]: active[1] = True; entry_z[1] = z
                sig = 'Entry_2'

            elif z < -1 and not active[1]:
                active[1] = True; entry_z[1] = z
                sig = 'Entry_1'

            else:
                live = [b for b in [3, 2, 1] if active[b]]
                sig  = f'In_{max(live)}' if live else 'Neutral'

        states.append(sig)
        n = sum(active.values())
        d26_lots.append(LOTS_D26 * n)
        spread_lots.append(LOTS_SPREAD * n)

    return states, d26_lots, spread_lots


# ─────────────────────────────────────────────
# 2. LOAD DATA
#    Sheet1 layout (two separate ticker blocks, each with Timestamp + Close):
#      Col A/B  : Timestamp, Close  → D26 (CLZ26)
#      Col D/E  : Timestamp, Close  → D26-D27 spread (CLZ26-Z27)
#    Adjust column indices below if your layout differs.
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

# --- D26 (CLZ26) ---
d26 = raw.iloc[:, [0, 1]].copy()
d26.columns = ['Timestamp', 'D26_Close']
d26['Timestamp'] = pd.to_datetime(d26['Timestamp'], dayfirst=True, errors='coerce')
d26['Timestamp'] = d26['Timestamp'].astype('datetime64[ns]')   # ensure numeric dtype for merge_asof
d26['D26_Close'] = pd.to_numeric(d26['D26_Close'], errors='coerce')
d26 = d26.dropna().sort_values('Timestamp').reset_index(drop=True)

# --- D26-D27 Calendar Spread (CLZ26-Z27) ---
spd = raw.iloc[:, [3, 4]].copy()
spd.columns = ['Timestamp', 'Spread_Close']
spd['Timestamp'] = pd.to_datetime(spd['Timestamp'], dayfirst=True, errors='coerce')
spd['Timestamp'] = spd['Timestamp'].astype('datetime64[ns]')   # ensure numeric dtype for merge_asof
spd['Spread_Close'] = pd.to_numeric(spd['Spread_Close'], errors='coerce')
spd = spd.dropna().sort_values('Timestamp').reset_index(drop=True)

# ─────────────────────────────────────────────
# 3. BUILD SYNTHETIC SPREAD PRICE  &  DAILY STATS
#    Spread_Value = MULT_D26 * D26 - MULT_SPREAD * (D26-D27)
# ─────────────────────────────────────────────
# Hourly merge
hourly = pd.merge_asof(d26, spd, on='Timestamp', direction='nearest')
hourly['Spread_Value'] = MULT_D26 * hourly['D26_Close'] - MULT_SPREAD * hourly['Spread_Close']
hourly['Date'] = hourly['Timestamp'].dt.date
hourly = hourly[hourly['Date'].apply(
    lambda d: d.weekday() < 5 and d not in EXCLUDED_DATES
)].reset_index(drop=True)

# Daily last price for rolling stats
daily_spd = (hourly.groupby('Date')['Spread_Value'].last()
             .reset_index().rename(columns={'Spread_Value': 'Daily_Spread'}))
daily_spd = daily_spd.sort_values('Date').reset_index(drop=True)

# Rolling 30-day mean, std, 14-day high/low (stop levels)
roll_mean, roll_std, stop_high, stop_low = [], [], [], []
for i in range(len(daily_spd)):
    win_start = max(0, i - ROLLING_WINDOW + 1)
    window    = daily_spd['Daily_Spread'].values[win_start: i + 1]
    if len(window) < ROLLING_WINDOW:
        roll_mean.append(np.nan); roll_std.append(np.nan)
    else:
        roll_mean.append(window.mean()); roll_std.append(window.std(ddof=1))
    # 14-day stop
    stop_win = daily_spd['Daily_Spread'].values[max(0, i - 13): i + 1]
    stop_high.append(stop_win.max())
    stop_low.append(stop_win.min())

daily_spd['Roll_Mean'] = roll_mean
daily_spd['Roll_Std']  = roll_std
daily_spd['Stop_High'] = stop_high
daily_spd['Stop_Low']  = stop_low

# ─────────────────────────────────────────────
# 4. MERGE STATS BACK → HOURLY  &  COMPUTE Z-SCORE
# ─────────────────────────────────────────────
hourly = hourly.join(
    daily_spd.set_index('Date')[['Roll_Mean', 'Roll_Std', 'Stop_High', 'Stop_Low']],
    on='Date'
).dropna(subset=['Roll_Mean', 'Roll_Std'])

hourly['Z_Score'] = (hourly['Spread_Value'] - hourly['Roll_Mean']) / hourly['Roll_Std']

sigs, dl, sl = compute_signals(hourly['Z_Score'].values)
hourly['Signal']      = sigs
hourly['D26_Lots']    = dl
hourly['Spread_Lots'] = sl

# ─────────────────────────────────────────────
# 5. FILTER LAST 60 WORKING DAYS
# ─────────────────────────────────────────────
def get_start_date(n_days, end_date):
    count, curr = 0, end_date
    while count < n_days:
        curr -= timedelta(days=1)
        if curr.weekday() < 5 and curr not in EXCLUDED_DATES:
            count += 1
    return curr

start_dt = get_start_date(DISPLAY_DAYS, hourly['Date'].max())
display  = hourly[hourly['Date'] >= start_dt].copy().reset_index(drop=True)

# ─────────────────────────────────────────────
# 6. TRADE SUMMARY
#    Step A: Walk full history BEFORE display window to find any
#            bands that are still open at the start of the window.
#            These "carry-in" positions are inserted as synthetic
#            Entry rows so every Exit in the summary has a matching entry.
# ─────────────────────────────────────────────

# --- Pre-window scan to initialise vault ---
pre_window = hourly[hourly['Date'] < start_dt].copy()

vault       = {1: None, 2: None, 3: None}   # {band: (row_series, entry_z)}
pre_entry_z = {1: None, 2: None, 3: None}

for _, row in pre_window.iterrows():
    sig = str(row['Signal'])
    if sig.startswith('Entry_'):
        band = int(sig.split('_')[1])
        if vault[band] is None:
            vault[band]       = row
            pre_entry_z[band] = row['Z_Score']
            # waterfall: deeper entry marks shallower bands too
            for b in range(band - 1, 0, -1):
                if vault[b] is None:
                    vault[b]       = row
                    pre_entry_z[b] = row['Z_Score']
    elif sig.startswith('Exit_'):
        band = int(sig.split('_')[1])
        vault[band]       = None
        pre_entry_z[band] = None

# Convert vault to spread-value dict for PnL (same shape as before)
# vault_val: {band: spread_value_at_entry}  or None
vault_val = {}
summary_rows = []

for band in [1, 2, 3]:
    if vault[band] is not None:
        entry_row = vault[band]
        vault_val[band] = entry_row['Spread_Value']
        # Emit a synthetic carry-in row marked as Entry so summary is complete
        r = entry_row.to_dict()
        r['D26_Lots']     = LOTS_D26
        r['Spread_Lots']  = LOTS_SPREAD
        r['Realized_PnL'] = 0
        r['Signal']       = f'Entry_{band}'   # ensure correct label
        summary_rows.append(r)
    else:
        vault_val[band] = None

# Also keep entry_z for exit condition continuity (already embedded in
# the running state machine — exits in display use pre-computed Signal column)

# --- Walk display window ---
for _, row in display.iterrows():
    sig = str(row['Signal'])
    if sig.startswith('Entry_'):
        band = int(sig.split('_')[1])
        if vault_val[band] is None:
            vault_val[band] = row['Spread_Value']
            r = row.to_dict()
            r['D26_Lots']     = LOTS_D26
            r['Spread_Lots']  = LOTS_SPREAD
            r['Realized_PnL'] = 0
            summary_rows.append(r)

    elif sig.startswith('Exit_'):
        band = int(sig.split('_')[1])
        if vault_val[band] is not None:
            en_val = vault_val[band]
            pnl    = (row['Spread_Value'] - en_val) * PNL_MULTIPLIER
            r = row.to_dict()
            r['D26_Lots']     = 0
            r['Spread_Lots']  = 0
            r['Realized_PnL'] = round(pnl, 2)
            summary_rows.append(r)
            vault_val[band]   = None

summary_df = pd.DataFrame(summary_rows)

# ─────────────────────────────────────────────
# 7. WRITE TO EXCEL
# ─────────────────────────────────────────────
with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    display.to_excel(writer, sheet_name=OUTPUT_SHEET, index=False, startrow=8)
    if not summary_df.empty:
        summary_df.to_excel(writer, sheet_name=SUMMARY_SHEET, index=False)

# Colour-code summary rows
wb     = load_workbook(INPUT_FILE)
ws_sum = wb[SUMMARY_SHEET]
for i, sig in enumerate(summary_df['Signal'], start=2):
    hex_c, white = BAND_COLOURS.get(sig, (None, False))
    if hex_c:
        for col in range(1, ws_sum.max_column + 1):
            cell      = ws_sum.cell(row=i, column=col)
            cell.fill = PatternFill(fill_type='solid', fgColor=hex_c)
            if white:
                cell.font = Font(color='FFFFFF')

wb.save(INPUT_FILE)
print(f"✓ WTI Crude spread analysis complete. Display window back to {start_dt}.")
print(f"  Spread formula : {MULT_D26}×D26  −  {MULT_SPREAD}×(D26-D27)")
print(f"  Rolling window : {ROLLING_WINDOW} days | Stop: 14-day High/Low")
print(f"  Entry bands    : -1 / -2 / -3 SD  |  Exit: entry_z + 1 SD")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_9384\876623746.py:105: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d26['Timestamp'] = pd.to_datetime(d26['Timestamp'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_9384\876623746.py:113: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spd['Timestamp'] = pd.to_datetime(spd['Timestamp'], dayfirst=True, errors='coerce')


✓ WTI Crude spread analysis complete. Display window back to 2026-01-22.
  Spread formula : 2000×D26  −  5000×(D26-D27)
  Rolling window : 30 days | Stop: 14-day High/Low
  Entry bands    : -1 / -2 / -3 SD  |  Exit: entry_z + 1 SD


In [8]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import timedelta, date

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE      = 'WTI_spreadanalysis.xlsx'
SHEET_NAME      = 'Sheet1'
OUTPUT_SHEET    = 'Spread_Analysis_v2'
SUMMARY_SHEET   = 'Trade_Summary'
EXCLUDED_DATES  = set() 
ROLLING_WINDOW  = 30 
DISPLAY_DAYS    = 60 
PNL_MULTIPLIER  = 1 

LOTS_D26        = 2 
LOTS_SPREAD     = 5 
MULT_D26        = 1000 * 2 
MULT_SPREAD     = 1000 * 5 

BAND_COLOURS = {
    'Entry_3':  ('7B0000', True),  'Exit_3':  ('C6E0B4', False),
    'Entry_2':  ('FF0000', True),  'Exit_2':  ('C6E0B4', False),
    'Entry_1':  ('FF9C00', True),  'Exit_1':  ('C6E0B4', False),
    'Entry_P3': ('7B0000', True),  'Exit_P3': ('C6E0B4', False),
    'Entry_P2': ('FF0000', True),  'Exit_P2': ('C6E0B4', False),
    'Entry_P1': ('FF9C00', True),  'Exit_P1': ('C6E0B4', False),
}

# ─────────────────────────────────────────────
# 1. STATE MACHINE (Handles +/- Bands & Multi-Events)
# ─────────────────────────────────────────────
def compute_signals(z_series):
    states, d26_lots, spread_lots = [], [], []
    active_neg = {1: False, 2: False, 3: False}; entry_z_neg = {1: None, 2: None, 3: None}
    active_pos = {1: False, 2: False, 3: False}; entry_z_pos = {1: None, 2: None, 3: None}

    for z in z_series:
        if np.isnan(z):
            states.append('Neutral'); d26_lots.append(0); spread_lots.append(0)
            continue

        bar_events = []

        # --- EXITS ---
        for b in [3, 2, 1]:
            if active_neg[b] and z >= (entry_z_neg[b] + 1.0):
                active_neg[b] = False; bar_events.append(f'Exit_{b}')
            if active_pos[b] and z <= (entry_z_pos[b] - 1.0):
                active_pos[b] = False; bar_events.append(f'Exit_P{b}')

        # --- ENTRIES ---
        if z < -1:
            for b in [3, 2, 1]:
                if z < -b and not active_neg[b]:
                    active_neg[b] = True; entry_z_neg[b] = z
                    bar_events.append(f'Entry_{b}')
        if z > 1:
            for b in [3, 2, 1]:
                if z > b and not active_pos[b]:
                    active_pos[b] = True; entry_z_pos[b] = z
                    bar_events.append(f'Entry_P{b}')

        if bar_events:
            sig = " | ".join(bar_events)
        else:
            l_neg = [b for b, a in active_neg.items() if a]
            l_pos = [b for b, a in active_pos.items() if a]
            if l_neg: sig = f'In_{max(l_neg)}'
            elif l_pos: sig = f'In_P{max(l_pos)}'
            else: sig = 'Neutral'

        states.append(sig)
        n_long, n_short = sum(active_neg.values()), sum(active_pos.values())
        net = n_long - n_short
        d26_lots.append(LOTS_D26 * net); spread_lots.append(LOTS_SPREAD * net)

    return states, d26_lots, spread_lots

# ─────────────────────────────────────────────
# 2. DATA LOAD & PREP
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

d26 = raw.iloc[:, [0, 1]].copy(); d26.columns = ['Timestamp', 'D26_Close']
d26['Timestamp'] = pd.to_datetime(d26['Timestamp'], dayfirst=True, errors='coerce')
d26['D26_Close'] = pd.to_numeric(d26['D26_Close'], errors='coerce')
d26 = d26.dropna().sort_values('Timestamp').reset_index(drop=True)

spd = raw.iloc[:, [3, 4]].copy(); spd.columns = ['Timestamp', 'Spread_Close']
spd['Timestamp'] = pd.to_datetime(spd['Timestamp'], dayfirst=True, errors='coerce')
spd['Spread_Close'] = pd.to_numeric(spd['Spread_Close'], errors='coerce')
spd = spd.dropna().sort_values('Timestamp').reset_index(drop=True)

hourly = pd.merge_asof(d26, spd, on='Timestamp', direction='nearest')
hourly['Spread_Value'] = MULT_D26 * hourly['D26_Close'] - MULT_SPREAD * hourly['Spread_Close']
hourly['Date'] = hourly['Timestamp'].dt.date

daily_spd = hourly.groupby('Date')['Spread_Value'].last().reset_index().rename(columns={'Spread_Value': 'Daily_Spread'})
daily_spd = daily_spd.sort_values('Date').reset_index(drop=True)

# Rolling Stats
rm, rs, sh, sl = [], [], [], []
for i in range(len(daily_spd)):
    win = daily_spd['Daily_Spread'].values[max(0, i-ROLLING_WINDOW+1): i+1]
    rm.append(win.mean() if len(win) >= ROLLING_WINDOW else np.nan)
    rs.append(win.std(ddof=1) if len(win) >= ROLLING_WINDOW else np.nan)
    s_win = daily_spd['Daily_Spread'].values[max(0, i-13): i+1]
    sh.append(s_win.max()); sl.append(s_win.min())

daily_spd['Roll_Mean'], daily_spd['Roll_Std'] = rm, rs
daily_spd['Stop_High'], daily_spd['Stop_Low'] = sh, sl

hourly = hourly.join(daily_spd.set_index('Date')[['Roll_Mean', 'Roll_Std', 'Stop_High', 'Stop_Low']], on='Date').dropna(subset=['Roll_Mean'])
hourly['Z_Score'] = (hourly['Spread_Value'] - hourly['Roll_Mean']) / hourly['Roll_Std']
sigs, dl, sl = compute_signals(hourly['Z_Score'].values)
hourly['Signal'], hourly['D26_Lots'], hourly['Spread_Lots'] = sigs, dl, sl

# ─────────────────────────────────────────────
# 3. FILTER DISPLAY WINDOW
# ─────────────────────────────────────────────
def get_start_date(n_days, end_date):
    count, curr = 0, end_date
    while count < n_days:
        curr -= timedelta(days=1)
        if curr.weekday() < 5 and curr not in EXCLUDED_DATES:
            count += 1
    return curr

hourly['Date'] = pd.to_datetime(hourly['Date']).dt.date
start_dt = get_start_date(DISPLAY_DAYS, hourly['Date'].max())
display = hourly[hourly['Date'] >= start_dt].copy().reset_index(drop=True)

# ─────────────────────────────────────────────
# 4. TRADE SUMMARY (Log Logic)
# ─────────────────────────────────────────────
pre_window = hourly[hourly['Date'] < start_dt].copy()
vault = {str(k): None for k in ['1','2','3','P1','P2','P3']}

for _, row in pre_window.iterrows():
    for s in [x.strip() for x in str(row['Signal']).split('|')]:
        if s.startswith('Entry_'): vault[s.split('_')[1]] = row
        elif s.startswith('Exit_'): vault[s.split('_')[1]] = None

vault_val, summary_rows = {}, []
for k, row in vault.items():
    if row is not None:
        vault_val[k] = row['Spread_Value']
        r = row.to_dict(); r['Signal'] = f'Entry_{k}'; r['Realized_PnL'] = 0
        summary_rows.append(r)
    else: vault_val[k] = None

for _, row in display.iterrows():
    for s in [x.strip() for x in str(row['Signal']).split('|')]:
        if s.startswith('Entry_'):
            bk = s.split('_')[1]
            if vault_val[bk] is None:
                vault_val[bk] = row['Spread_Value']
                r = row.to_dict(); r['Signal'] = s; r['Realized_PnL'] = 0
                summary_rows.append(r)
        elif s.startswith('Exit_'):
            bk = s.split('_')[1]
            if vault_val[bk] is not None:
                pnl = (vault_val[bk] - row['Spread_Value']) if 'P' in bk else (row['Spread_Value'] - vault_val[bk])
                r = row.to_dict(); r['Signal'] = s; r['Realized_PnL'] = round(pnl * PNL_MULTIPLIER, 2)
                r['D26_Lots'], r['Spread_Lots'] = 0, 0
                summary_rows.append(r); vault_val[bk] = None

summary_df = pd.DataFrame(summary_rows)

# ─────────────────────────────────────────────
# 5. OUTPUT & STYLING
# ─────────────────────────────────────────────
with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    display.to_excel(writer, sheet_name=OUTPUT_SHEET, index=False, startrow=8)
    if not summary_df.empty: summary_df.to_excel(writer, sheet_name=SUMMARY_SHEET, index=False)

wb = load_workbook(INPUT_FILE)
if SUMMARY_SHEET in wb.sheetnames:
    ws = wb[SUMMARY_SHEET]
    for i, sig in enumerate(summary_df['Signal'], start=2):
        hex_c, white = BAND_COLOURS.get(sig, (None, False))
        if hex_c:
            for col in range(1, ws.max_column + 1):
                ws.cell(row=i, column=col).fill = PatternFill(fill_type='solid', fgColor=hex_c)
                if white: ws.cell(row=i, column=col).font = Font(color='FFFFFF')
wb.save(INPUT_FILE)
print(f"✓ Complete. Analysis back to {start_dt}.")

C:\Users\Axxela\AppData\Local\Temp\ipykernel_22344\766648127.py:90: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d26['Timestamp'] = pd.to_datetime(d26['Timestamp'], dayfirst=True, errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_22344\766648127.py:95: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spd['Timestamp'] = pd.to_datetime(spd['Timestamp'], dayfirst=True, errors='coerce')


✓ Complete. Analysis back to 2026-01-22.


In [5]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from datetime import timedelta, date


# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'AEM_XAU_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
SUMMARY_SHEET  = 'Trade_Summary'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 20        # 20 trading days lookback for daily regression
DISPLAY_DAYS   = 60

BAND_COLOURS = {
    'Entry_4':  ('7B0000', True),  'Exit_4':   ('C6E0B4', False),
    'Entry_3':  ('C00000', True),  'Exit_3':   ('C6E0B4', False),
    'Entry_2':  ('FF0000', True),  'Exit_2':   ('C6E0B4', False),
    'Entry_1':  ('FF9C00', True),  'Exit_1':   ('C6E0B4', False),
}

LOTS_AEM = 10
LOTS_XAU = 1
BAND_THRESHOLDS = {1: -1, 2: -2, 3: -3, 4: -4}

# ─────────────────────────────────────────────
# 1. STATE MACHINE
# ─────────────────────────────────────────────
def compute_signals(z_series):
    signals  = []
    aem_lots = []
    xau_lots = []
    active  = {1: False, 2: False, 3: False, 4: False}
    entry_z = {1: None,  2: None,  3: None,  4: None}

    for z in z_series:
        if np.isnan(z):
            signals.append('Neutral'); aem_lots.append(0); xau_lots.append(0)
            continue
        bar_events = []
        for b in [4, 3, 2, 1]:
            if active[b] and z >= (entry_z[b] + 1.0):
                active[b] = False; entry_z[b] = None
                bar_events.append(f'Exit_{b}')
        for b in [4, 3, 2, 1]:
            if not active[b] and z <= BAND_THRESHOLDS[b]:
                active[b] = True; entry_z[b] = z
                bar_events.append(f'Entry_{b}')
        if bar_events:
            sig_str = " | ".join(bar_events)
        else:
            live = [b for b in [4, 3, 2, 1] if active[b]]
            sig_str = f'In_{max(live)}' if live else 'Neutral'
        signals.append(sig_str)
        aem_lots.append(LOTS_AEM)
        xau_lots.append(LOTS_XAU)

    return signals, aem_lots, xau_lots

# ─────────────────────────────────────────────
# 2. DATA LOAD
#    Col 0,1 → XAU DateTime, XAU_Close
#    Col 3,4 → AEM DateTime, AEM_Close
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)
print(f"Raw shape: {raw.shape}")

xau = raw.iloc[:, [0, 1]].copy()
xau.columns = ['DateTime', 'XAU_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], errors='coerce')
xau['XAU_Close'] = pd.to_numeric(xau['XAU_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)

aem = raw.iloc[:, [3, 4]].copy()
aem.columns = ['DateTime', 'AEM_Close']
aem['DateTime']  = pd.to_datetime(aem['DateTime'], errors='coerce')
aem['AEM_Close'] = pd.to_numeric(aem['AEM_Close'], errors='coerce')
aem = aem.dropna().sort_values('DateTime').reset_index(drop=True)

print(f"XAU rows: {len(xau)} | {xau['DateTime'].min()} → {xau['DateTime'].max()}")
print(f"AEM rows: {len(aem)} | {aem['DateTime'].min()} → {aem['DateTime'].max()}")

# ─────────────────────────────────────────────
# 3. DAILY REGRESSION  (last price per calendar date)
# ─────────────────────────────────────────────
xau['_date'] = xau['DateTime'].dt.date
aem['_date'] = aem['DateTime'].dt.date

xau_d = xau.groupby('_date')['XAU_Close'].last().reset_index()
aem_d = aem.groupby('_date')['AEM_Close'].last().reset_index()

daily = xau_d.merge(aem_d, on='_date', how='inner')
daily.columns = ['Date', 'XAU_Close', 'AEM_Close']
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d.date() not in EXCLUDED_DATES)
].reset_index(drop=True)
print(f"Daily rows: {len(daily)} | Rolling window: {ROLLING_WINDOW}")

slopes, ints, ms, ss = [], [], [], []
for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        for lst in [slopes, ints, ms, ss]: lst.append(np.nan)
        continue
    x = daily['XAU_Close'].values[i - ROLLING_WINDOW + 1 : i + 1].astype(float)
    y = daily['AEM_Close'].values[i - ROLLING_WINDOW + 1 : i + 1].astype(float)
    m, c = np.polyfit(x, y, 1)
    res  = y - (m * x + c)
    slopes.append(m); ints.append(c); ms.append(res.mean()); ss.append(res.std(ddof=1))

daily['Slope'], daily['Intercept'], daily['Res_Mean'], daily['Res_Std'] = slopes, ints, ms, ss
print(f"Daily rows with valid Slope: {daily['Slope'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. HOURLY MERGE  — join regression params by date
# ─────────────────────────────────────────────
hourly = pd.merge_asof(xau, aem, on='DateTime', direction='nearest')
hourly['_date'] = pd.to_datetime(hourly['DateTime'].dt.date)

# Join daily regression params — both keys must be datetime
daily_lkp = daily.set_index('Date')[['Slope', 'Intercept', 'Res_Mean', 'Res_Std']]
hourly = hourly.join(daily_lkp, on='_date').dropna(subset=['Slope', 'Res_Std'])
print(f"Hourly rows after regression join: {len(hourly)}")

hourly['Residual'] = hourly['AEM_Close'] - (
    hourly['Slope'] * hourly['XAU_Close'] + hourly['Intercept']
)
hourly['Z_Score'] = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']

z_valid = hourly['Z_Score'].dropna()
print(f"Z_Score range: {z_valid.min():.3f} → {z_valid.max():.3f}")

sigs, al, xl = compute_signals(hourly['Z_Score'].values)
hourly['Signal'], hourly['AEM_Lots'], hourly['XAU_Lots'] = sigs, al, xl

# ─────────────────────────────────────────────
# 5. DISPLAY WINDOW
# ─────────────────────────────────────────────
start_dt = hourly['_date'].max() - timedelta(days=DISPLAY_DAYS)
display  = hourly[hourly['_date'] >= start_dt].copy().reset_index(drop=True)
print(f"Display rows (last {DISPLAY_DAYS} days): {len(display)}")

# ─────────────────────────────────────────────
# 6. SUMMARY — Entries & Exits with PnL
# ─────────────────────────────────────────────
summary_rows = []
vault        = {1: None,  2: None,  3: None,  4: None}
active_sum   = {1: False, 2: False, 3: False, 4: False}
entry_z_sum  = {1: None,  2: None,  3: None,  4: None}

for _, row in display.iterrows():
    z = row['Z_Score']
    if np.isnan(z):
        continue
    for b in [4, 3, 2, 1]:
        if active_sum[b] and z >= (entry_z_sum[b] + 1.0):
            en_a, en_x, en_slope = vault[b]
            pnl_aem = (row['AEM_Close'] - en_a) * LOTS_AEM * (1 / en_slope) *100
            pnl_xau = (en_x - row['XAU_Close']) * LOTS_XAU
            r_exit = row.to_dict()
            r_exit['Band']         = b
            r_exit['Event']        = f'Exit_{b}'
            r_exit['Realized_PnL'] = round(pnl_aem + pnl_xau, 2)
            summary_rows.append(r_exit)
            active_sum[b] = False; vault[b] = None
    for b in [4, 3, 2, 1]:
        if not active_sum[b] and z <= BAND_THRESHOLDS[b]:
            active_sum[b]  = True
            entry_z_sum[b] = z
            vault[b] = (row['AEM_Close'], row['XAU_Close'], row['Slope'])
            r_entry = row.to_dict()
            r_entry['Band']         = b
            r_entry['Event']        = f'Entry_{b}'
            r_entry['Realized_PnL'] = 0.0
            summary_rows.append(r_entry)

summary_df = pd.DataFrame(summary_rows)
print(f"Summary events logged: {len(summary_df)}")

# ─────────────────────────────────────────────
# 7. EXCEL OUTPUT
# ─────────────────────────────────────────────
if display.empty:
    print("✗ FATAL: display is empty — nothing written.")
else:
    sigs, al, xl = compute_signals(display['Z_Score'].values)
    display['Signal']   = sigs
    display['AEM_Lots'] = al
    display['XAU_Lots'] = xl

    # Drop internal helper column before writing
    display = display.drop(columns=['_date'], errors='ignore')
    if not summary_df.empty:
        summary_df = summary_df.drop(columns=['_date'], errors='ignore')

    with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        display.to_excel(writer, sheet_name=OUTPUT_SHEET, index=False, startrow=8)
        if not summary_df.empty:
            summary_df.to_excel(writer, sheet_name=SUMMARY_SHEET, index=False)

    wb     = load_workbook(INPUT_FILE)
    ws_sum = wb[SUMMARY_SHEET] if SUMMARY_SHEET in wb.sheetnames else None

    if ws_sum is not None and not summary_df.empty:
        for i, event in enumerate(summary_df['Event'], start=2):
            hex_c, white = BAND_COLOURS.get(event, (None, False))
            if hex_c:
                for col in range(1, ws_sum.max_column + 1):
                    cell = ws_sum.cell(row=i, column=col)
                    cell.fill = PatternFill(fill_type='solid', fgColor=hex_c)
                    if white:
                        cell.font = Font(color='FFFFFF')

    wb.save(INPUT_FILE)
    print(f"✓ Done. {len(display)} rows → '{OUTPUT_SHEET}' | {len(summary_df)} events → '{SUMMARY_SHEET}'.")

Raw shape: (1795, 5)
XAU rows: 1794 | 2025-12-31 23:30:00 → 2026-04-18 03:30:00
AEM rows: 1174 | 2025-12-31 23:30:00 → 2026-04-18 05:30:00
Daily rows: 75 | Rolling window: 20
Daily rows with valid Slope: 56
Hourly rows after regression join: 1301
Z_Score range: -3.683 → 2.228
Display rows (last 60 days): 998
Summary events logged: 10


C:\Users\Axxela\AppData\Local\Temp\ipykernel_23032\3103326514.py:74: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_23032\3103326514.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  aem['DateTime']  = pd.to_datetime(aem['DateTime'], errors='coerce')


✓ Done. 998 rows → 'Spread_Analysis_v2' | 10 events → 'Trade_Summary'.


In [9]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from datetime import timedelta, date


# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
INPUT_FILE     = 'GDMN_UGL_spreadanalysis.xlsx'
SHEET_NAME     = 'Sheet1'
OUTPUT_SHEET   = 'Spread_Analysis_v2'
SUMMARY_SHEET  = 'Trade_Summary'
EXCLUDED_DATES = {date(2026, 4, 3)}
ROLLING_WINDOW = 20        # 20 trading days lookback for daily regression
DISPLAY_DAYS   = 60

BAND_COLOURS = {
    'Entry_4':  ('7B0000', True),  'Exit_4':   ('C6E0B4', False),
    'Entry_3':  ('C00000', True),  'Exit_3':   ('C6E0B4', False),
    'Entry_2':  ('FF0000', True),  'Exit_2':   ('C6E0B4', False),
    'Entry_1':  ('FF9C00', True),  'Exit_1':   ('C6E0B4', False),
}

LOTS_UGL = 1
LOTS_GDMN = 1
BAND_THRESHOLDS = {1: -1, 2: -2, 3: -3, 4: -4}

# ─────────────────────────────────────────────
# 1. STATE MACHINE
# ─────────────────────────────────────────────
def compute_signals(z_series):
    signals  = []
    aem_lots = []
    xau_lots = []
    active  = {1: False, 2: False, 3: False, 4: False}
    entry_z = {1: None,  2: None,  3: None,  4: None}

    for z in z_series:
        if np.isnan(z):
            signals.append('Neutral'); aem_lots.append(0); xau_lots.append(0)
            continue
        bar_events = []
        for b in [4, 3, 2, 1]:
            if active[b] and z >= (entry_z[b] + 1.0):
                active[b] = False; entry_z[b] = None
                bar_events.append(f'Exit_{b}')
        for b in [4, 3, 2, 1]:
            if not active[b] and z <= BAND_THRESHOLDS[b]:
                active[b] = True; entry_z[b] = z
                bar_events.append(f'Entry_{b}')
        if bar_events:
            sig_str = " | ".join(bar_events)
        else:
            live = [b for b in [4, 3, 2, 1] if active[b]]
            sig_str = f'In_{max(live)}' if live else 'Neutral'
        signals.append(sig_str)
        aem_lots.append(LOTS_UGL)
        xau_lots.append(LOTS_GDMN)

    return signals, aem_lots, xau_lots

# ─────────────────────────────────────────────
# 2. DATA LOAD
#    Col 0,1 → XAU DateTime, XAU_Close
#    Col 3,4 → AEM DateTime, AEM_Close
# ─────────────────────────────────────────────
raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=0)
print(f"Raw shape: {raw.shape}")

xau = raw.iloc[:, [1, 2]].copy()
xau.columns = ['DateTime', 'UGL_Close']
xau['DateTime']  = pd.to_datetime(xau['DateTime'], errors='coerce')
xau['UGL_Close'] = pd.to_numeric(xau['UGL_Close'], errors='coerce')
xau = xau.dropna().sort_values('DateTime').reset_index(drop=True)

aem = raw.iloc[:, [3, 4]].copy()
aem.columns = ['DateTime', 'GDMN_Close']
aem['DateTime']  = pd.to_datetime(aem['DateTime'], errors='coerce')
aem['GDMN_Close'] = pd.to_numeric(aem['GDMN_Close'], errors='coerce')
aem = aem.dropna().sort_values('DateTime').reset_index(drop=True)

print(f"GDMN rows: {len(aem)} | {aem['DateTime'].min()} → {aem['DateTime'].max()}")
print(f"UGL rows: {len(xau)} | {xau['DateTime'].min()} → {xau['DateTime'].max()}")

# ─────────────────────────────────────────────
# 3. DAILY REGRESSION  (last price per calendar date)
# ─────────────────────────────────────────────
xau['_date'] = xau['DateTime'].dt.date
aem['_date'] = aem['DateTime'].dt.date

xau_d = xau.groupby('_date')['UGL_Close'].last().reset_index()
aem_d = aem.groupby('_date')['GDMN_Close'].last().reset_index()

daily = xau_d.merge(aem_d, on='_date', how='inner')
daily.columns = ['Date', 'UGL_Close', 'GDMN_Close']
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily[
    daily['Date'].apply(lambda d: d.weekday() < 5 and d.date() not in EXCLUDED_DATES)
].reset_index(drop=True)
print(f"Daily rows: {len(daily)} | Rolling window: {ROLLING_WINDOW}")

slopes, ints, ms, ss = [], [], [], []
for i in range(len(daily)):
    if i < ROLLING_WINDOW - 1:
        for lst in [slopes, ints, ms, ss]: lst.append(np.nan)
        continue
    x = daily['UGL_Close'].values[i - ROLLING_WINDOW + 1 : i + 1].astype(float)
    y = daily['GDMN_Close'].values[i - ROLLING_WINDOW + 1 : i + 1].astype(float)
    m, c = np.polyfit(x, y, 1)
    res  = y - (m * x + c)
    slopes.append(m); ints.append(c); ms.append(res.mean()); ss.append(res.std(ddof=1))

daily['Slope'], daily['Intercept'], daily['Res_Mean'], daily['Res_Std'] = slopes, ints, ms, ss
print(f"Daily rows with valid Slope: {daily['Slope'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. HOURLY MERGE  — join regression params by date
# ─────────────────────────────────────────────
hourly = pd.merge_asof(xau, aem, on='DateTime', direction='nearest')
hourly['_date'] = pd.to_datetime(hourly['DateTime'].dt.date)

# Join daily regression params — both keys must be datetime
daily_lkp = daily.set_index('Date')[['Slope', 'Intercept', 'Res_Mean', 'Res_Std']]
hourly = hourly.join(daily_lkp, on='_date').dropna(subset=['Slope', 'Res_Std'])
print(f"Hourly rows after regression join: {len(hourly)}")

hourly['Residual'] = hourly['GDMN_Close'] - (
    hourly['Slope'] * hourly['UGL_Close'] + hourly['Intercept']
)
hourly['Z_Score'] = (hourly['Residual'] - hourly['Res_Mean']) / hourly['Res_Std']

z_valid = hourly['Z_Score'].dropna()
print(f"Z_Score range: {z_valid.min():.3f} → {z_valid.max():.3f}")

sigs, al, xl = compute_signals(hourly['Z_Score'].values)
hourly['Signal'], hourly['GDMN_Lots'], hourly['UGL_Lots'] = sigs, al, xl

# ─────────────────────────────────────────────
# 5. DISPLAY WINDOW
# ─────────────────────────────────────────────
start_dt = hourly['_date'].max() - timedelta(days=DISPLAY_DAYS)
display  = hourly[hourly['_date'] >= start_dt].copy().reset_index(drop=True)
print(f"Display rows (last {DISPLAY_DAYS} days): {len(display)}")

# ─────────────────────────────────────────────
# 6. SUMMARY — Entries & Exits with PnL
# ─────────────────────────────────────────────
summary_rows = []
vault        = {1: None,  2: None,  3: None,  4: None}
active_sum   = {1: False, 2: False, 3: False, 4: False}
entry_z_sum  = {1: None,  2: None,  3: None,  4: None}

for _, row in display.iterrows():
    z = row['Z_Score']
    if np.isnan(z):
        continue
    for b in [4, 3, 2, 1]:
        if active_sum[b] and z >= (entry_z_sum[b] + 1.0):
            en_a, en_x, en_slope = vault[b]
            pnl_aem = (row['GDMN_Close'] - en_a) * LOTS_GDMN * (1 / en_slope)
            pnl_xau = (en_x - row['UGL_Close']) * LOTS_UGL
            r_exit = row.to_dict()
            r_exit['Band']         = b
            r_exit['Event']        = f'Exit_{b}'
            r_exit['Realized_PnL'] = round(pnl_aem + pnl_xau, 2)
            summary_rows.append(r_exit)
            active_sum[b] = False; vault[b] = None
    for b in [4, 3, 2, 1]:
        if not active_sum[b] and z <= BAND_THRESHOLDS[b]:
            active_sum[b]  = True
            entry_z_sum[b] = z
            vault[b] = (row['GDMN_Close'], row['UGL_Close'], row['Slope'])
            r_entry = row.to_dict()
            r_entry['Band']         = b
            r_entry['Event']        = f'Entry_{b}'
            r_entry['Realized_PnL'] = 0.0
            summary_rows.append(r_entry)

summary_df = pd.DataFrame(summary_rows)
print(f"Summary events logged: {len(summary_df)}")

# ─────────────────────────────────────────────
# 7. EXCEL OUTPUT
# ─────────────────────────────────────────────
if display.empty:
    print("✗ FATAL: display is empty — nothing written.")
else:
    sigs, al, xl = compute_signals(display['Z_Score'].values)
    display['Signal']   = sigs
    display['GDMN_Lots'] = al
    display['UGL_Lots'] = xl

    # Drop internal helper column before writing
    display = display.drop(columns=['_date'], errors='ignore')
    if not summary_df.empty:
        summary_df = summary_df.drop(columns=['_date'], errors='ignore')

    with pd.ExcelWriter(INPUT_FILE, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        display.to_excel(writer, sheet_name=OUTPUT_SHEET, index=False, startrow=8)
        if not summary_df.empty:
            summary_df.to_excel(writer, sheet_name=SUMMARY_SHEET, index=False)

    wb     = load_workbook(INPUT_FILE)
    ws_sum = wb[SUMMARY_SHEET] if SUMMARY_SHEET in wb.sheetnames else None

    if ws_sum is not None and not summary_df.empty:
        for i, event in enumerate(summary_df['Event'], start=2):
            hex_c, white = BAND_COLOURS.get(event, (None, False))
            if hex_c:
                for col in range(1, ws_sum.max_column + 1):
                    cell = ws_sum.cell(row=i, column=col)
                    cell.fill = PatternFill(fill_type='solid', fgColor=hex_c)
                    if white:
                        cell.font = Font(color='FFFFFF')

    wb.save(INPUT_FILE)
    print(f"✓ Done. {len(display)} rows → '{OUTPUT_SHEET}' | {len(summary_df)} events → '{SUMMARY_SHEET}'.")

Raw shape: (1156, 5)
GDMN rows: 1047 | 2025-12-31 23:30:00 → 2026-04-17 00:30:00
UGL rows: 1155 | 2025-12-31 23:30:00 → 2026-04-17 00:30:00
Daily rows: 75 | Rolling window: 20
Daily rows with valid Slope: 56
Hourly rows after regression join: 810
Z_Score range: -4.411 → 2.925
Display rows (last 60 days): 616
Summary events logged: 16


C:\Users\Axxela\AppData\Local\Temp\ipykernel_23032\2052675165.py:74: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  xau['DateTime']  = pd.to_datetime(xau['DateTime'], errors='coerce')
C:\Users\Axxela\AppData\Local\Temp\ipykernel_23032\2052675165.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  aem['DateTime']  = pd.to_datetime(aem['DateTime'], errors='coerce')


✓ Done. 616 rows → 'Spread_Analysis_v2' | 16 events → 'Trade_Summary'.
